In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ============================================================
# 07_locked_swin_ablation_development_only.ipynb
# COMPLETE SINGLE-CELL CODE
# ============================================================
#
# Purpose
# -------
# Perform an EXPLANATORY 2x2 ablation of the already locked
# representative Swin model:
#
#                    Ordinal OFF       Ordinal ON
# Weighted CE        A1                A0 (FULL LOCKED MODEL)
# Unweighted CE      A3                A2
#
# A0:
#   weighted CE + ordinal-distance loss
#   alpha = 1
#   lambda = 0.10
#
# A1:
#   weighted CE only
#
# A2:
#   unweighted CE + ordinal-distance loss
#   alpha = 1
#   lambda = 0.10
#
# A3:
#   unweighted CE only
#
# IMPORTANT
# ---------
# A0 was already locked in Notebook 06 BEFORE this ablation.
# This notebook is explanatory only.
#
# The ablation results CANNOT:
#   - change the locked Swin model
#   - change its loss
#   - change the aggregation rule
#   - change the patient reference
#   - alter the later final-test model
#
# A0 is NOT retrained here.
# Its uniform-runtime OOF predictions are loaded directly
# from Notebook 06.
#
# A1, A2, A3 are newly trained with exactly the same:
#   - frozen 5 patient folds
#   - 3 seeds: 42, 2025, 2026
#   - Swin backbone
#   - image transforms
#   - optimizer
#   - LR schedule
#   - TTA
#   - early stopping
#   - checkpoint hierarchy
#   - runtime configuration
#
# Primary reference:
#   modal_low
#
# Locked patient aggregation:
#   mean_probability_argmax
#
# Official test:
#   NEVER defined, loaded, or evaluated.
#
# New workload:
#   3 ablations x 3 seeds x 5 folds = 45 fold trainings
#
# Resume:
#   - completed ablation-seed experiments are reused
#   - completed folds are reused
#   - interrupted folds resume from last completed epoch
# ============================================================


# ============================================================
# 0. Install dependencies
# ============================================================

%pip -q install timm==1.0.28 scikit-learn==1.6.1 pandas==2.2.2 Pillow==11.3.0


# ============================================================
# 1. Imports
# ============================================================

import gc
import io
import os
import json
import math
import copy
import time
import random
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.metrics import (
    cohen_kappa_score,
    mean_absolute_error,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

from IPython.display import display


ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")


# ============================================================
# 2. Development-only safety check
# ============================================================

FORBIDDEN_TEST_OBJECTS = [
    "TEST_PATH",
    "TEST_ROOT",
    "TEST_DIR",
    "OFFICIAL_TEST_PATH",
    "OFFICIAL_TEST_ROOT",
    "test_df",
    "test_dataset",
    "test_loader",
    "test_predictions",
    "test_patient_predictions",
    "test_fold_prob_dfs",
    "all_test_fold_prob_df",
    "ensemble_test_df",
    "ensemble_test_patient_df",
    "ensemble_test_metrics",
    "ensemble_test_patient_metrics",
]

present_test_objects = [
    name
    for name in FORBIDDEN_TEST_OBJECTS
    if name in globals()
]

if present_test_objects:
    raise RuntimeError(
        "Official-test objects exist in the current runtime: "
        f"{present_test_objects}\n"
        "Restart the runtime and run only this cell."
    )

print("Development-only safety check passed.")
print("Official LIMUC test data are not defined or loaded.")


# ============================================================
# 3. Mount Google Drive
# ============================================================

from google.colab import drive

(drive.mount(
    "/content/drive",
    force_remount=True,
) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)


# ============================================================
# 4. Paths
# ============================================================

DEVELOPMENT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/train_and_validation_sets')
)

NOTEBOOK01_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/01_vit_loss_screening_development_only')
)

NOTEBOOK03_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/03_patient_aggregation_and_reference_sensitivity')
)

NOTEBOOK05_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/05_swin_loss_multiseed_confirmation_uniform_runtime_v1')
)

NOTEBOOK06_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/06_backbone_benchmark_development_only_v1')
)

OUTPUT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/07_locked_swin_ablation_development_only_v1')
)

MANIFEST_PATH = (
    NOTEBOOK01_ROOT
    / "development_image_manifest_with_folds.csv"
)

FOLD_ASSIGNMENT_PATH = (
    NOTEBOOK01_ROOT
    / "patient_fold_assignment.csv"
)

REFERENCE_AUDIT_PATH = (
    NOTEBOOK01_ROOT
    / "patient_reference_rule_sensitivity.csv"
)

LOCKED_AGGREGATION_PATH = (
    NOTEBOOK03_ROOT
    / "locked_patient_aggregation_rule.json"
)

NOTEBOOK06_PLAN_PATH = (
    NOTEBOOK06_ROOT
    / "analysis_plan_frozen_before_backbone_benchmark.json"
)

LOCKED_BACKBONE_PATH = (
    NOTEBOOK06_ROOT
    / "locked_representative_backbone.json"
)

NOTEBOOK06_DECISION_PATH = (
    NOTEBOOK06_ROOT
    / "backbone_benchmark_decision_summary.json"
)

required_paths = [
    DEVELOPMENT_ROOT,
    NOTEBOOK01_ROOT,
    NOTEBOOK03_ROOT,
    NOTEBOOK05_ROOT,
    NOTEBOOK06_ROOT,
    MANIFEST_PATH,
    FOLD_ASSIGNMENT_PATH,
    REFERENCE_AUDIT_PATH,
    LOCKED_AGGREGATION_PATH,
    NOTEBOOK06_PLAN_PATH,
    LOCKED_BACKBONE_PATH,
    NOTEBOOK06_DECISION_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path does not exist:\n{required_path}"
        )

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nDevelopment root:", DEVELOPMENT_ROOT)
print("Notebook 06 root:", NOTEBOOK06_ROOT)
print("Ablation output:", OUTPUT_ROOT)


# ============================================================
# 5. Atomic I/O and hash helpers
# ============================================================

DRIVE_IO_MAX_RETRIES = 5
DRIVE_IO_INITIAL_DELAY_SECONDS = 1.0


def temporary_path(
    path: Path,
):
    return path.with_name(
        path.name + ".temporary"
    )


def retry_delay(
    attempt_index,
    initial_delay,
    maximum_delay=8.0,
):
    return min(
        maximum_delay,
        initial_delay
        * (attempt_index + 1),
    )


def atomic_json_save(
    payload,
    path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        temp_path = temporary_path(
            path
        )

        try:
            if temp_path.exists():
                temp_path.unlink()

            with open(
                temp_path,
                "w",
                encoding="utf-8",
            ) as file:
                json.dump(
                    payload,
                    file,
                    indent=2,
                    default=str,
                )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            print(
                "JSON write retry",
                attempt_index + 1,
                "/",
                DRIVE_IO_MAX_RETRIES,
                ":",
                path,
            )

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save JSON:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def atomic_csv_save(
    dataframe,
    path,
    index=False,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        temp_path = temporary_path(
            path
        )

        try:
            if temp_path.exists():
                temp_path.unlink()

            dataframe.to_csv(
                temp_path,
                index=index,
            )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            print(
                "CSV write retry",
                attempt_index + 1,
                "/",
                DRIVE_IO_MAX_RETRIES,
                ":",
                path,
            )

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save CSV:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def atomic_torch_save(
    payload,
    path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        temp_path = temporary_path(
            path
        )

        try:
            if temp_path.exists():
                temp_path.unlink()

            torch.save(
                payload,
                temp_path,
            )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            print(
                "Checkpoint write retry",
                attempt_index + 1,
                "/",
                DRIVE_IO_MAX_RETRIES,
                ":",
                path,
            )

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save checkpoint:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def safe_torch_load(
    path,
    map_location="cpu",
):
    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        try:
            try:
                return torch.load(
                    path,
                    map_location=map_location,
                    weights_only=False,
                )

            except TypeError:
                return torch.load(
                    path,
                    map_location=map_location,
                )

        except OSError as error:
            last_error = error

            print(
                "Checkpoint read retry",
                attempt_index + 1,
                "/",
                DRIVE_IO_MAX_RETRIES,
                ":",
                path,
            )

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not load checkpoint:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def file_sha256(
    path,
):
    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            hasher.update(
                block
            )

    return hasher.hexdigest()


def dictionary_sha256(
    dictionary,
):
    encoded = json.dumps(
        dictionary,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


def safe_package_version(
    package_name,
):
    try:
        return version(
            package_name
        )

    except PackageNotFoundError:
        return None


for temporary_file in OUTPUT_ROOT.rglob(
    "*.temporary"
):
    try:
        temporary_file.unlink()
    except Exception:
        pass


# ============================================================
# 6. Load locked decisions from Notebook 06
# ============================================================

with open(
    NOTEBOOK06_PLAN_PATH,
    "r",
    encoding="utf-8",
) as file:
    notebook06_plan = json.load(
        file
    )

with open(
    LOCKED_BACKBONE_PATH,
    "r",
    encoding="utf-8",
) as file:
    locked_backbone = json.load(
        file
    )

with open(
    LOCKED_AGGREGATION_PATH,
    "r",
    encoding="utf-8",
) as file:
    locked_aggregation = json.load(
        file
    )

PRIMARY_REFERENCE_RULE = "modal_low"

LOCKED_AGGREGATION_RULE = (
    "mean_probability_argmax"
)

LOCKED_BACKBONE_NAME = "Swin"

if (
    locked_backbone.get(
        "locked_representative_backbone"
    )
    != LOCKED_BACKBONE_NAME
):
    raise RuntimeError(
        "Notebook 06 did not lock Swin as the "
        "representative backbone."
    )

if (
    locked_aggregation.get(
        "locked_aggregation_rule"
    )
    != LOCKED_AGGREGATION_RULE
):
    raise RuntimeError(
        "Notebook 03 did not lock "
        "mean_probability_argmax."
    )

if (
    locked_aggregation.get(
        "primary_reference_rule"
    )
    != PRIMARY_REFERENCE_RULE
):
    raise RuntimeError(
        "Notebook 03 did not use modal_low."
    )

LOCKED_SWIN_CONFIG = copy.deepcopy(
    locked_backbone[
        "locked_configuration"
    ]
)

SWIN_MODEL_NAME = str(
    LOCKED_SWIN_CONFIG[
        "model_name"
    ]
)

if (
    LOCKED_SWIN_CONFIG[
        "representative_configuration"
    ]
    !=
    "swin_L3_expected_ordinal_distance_a1_w010"
):
    raise RuntimeError(
        "Unexpected locked Swin configuration."
    )

if not np.isclose(
    float(
        LOCKED_SWIN_CONFIG[
            "ordinal_alpha"
        ]
    ),
    1.0,
):
    raise RuntimeError(
        "Locked Swin alpha is not 1."
    )

if not np.isclose(
    float(
        LOCKED_SWIN_CONFIG[
            "ordinal_loss_weight"
        ]
    ),
    0.10,
):
    raise RuntimeError(
        "Locked Swin ordinal weight is not 0.10."
    )

training_config = (
    notebook06_plan[
        "training_config"
    ]
)

print(
    "\nVerified locked backbone:",
    LOCKED_BACKBONE_NAME,
)

print(
    "Verified locked model:",
    SWIN_MODEL_NAME,
)

print(
    "Verified locked configuration:",
    LOCKED_SWIN_CONFIG[
        "representative_configuration"
    ],
)

print(
    "Verified primary reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Verified patient aggregation:",
    LOCKED_AGGREGATION_RULE,
)


# ============================================================
# 7. Scientific training configuration
# ============================================================

PROTOCOL_VERSION = (
    "locked_swin_factorial_ablation_development_only_v1"
)

NUM_CLASSES = int(
    training_config[
        "num_classes"
    ]
)

NUM_FOLDS = int(
    training_config[
        "num_folds"
    ]
)

IMAGE_SIZE = int(
    training_config[
        "image_size"
    ]
)

TRAIN_BATCH_SIZE = int(
    training_config[
        "train_batch_size"
    ]
)

GRADIENT_ACCUMULATION_STEPS = int(
    training_config[
        "gradient_accumulation_steps"
    ]
)

EFFECTIVE_TRAIN_BATCH_SIZE = int(
    training_config[
        "effective_train_batch_size"
    ]
)

VALIDATION_BATCH_SIZE = int(
    training_config[
        "validation_batch_size"
    ]
)

RUNTIME_NUM_WORKERS = 0

EPOCHS = int(
    training_config[
        "epochs"
    ]
)

LEARNING_RATE = float(
    training_config[
        "learning_rate"
    ]
)

MIN_LR_RATIO = float(
    training_config[
        "min_lr_ratio"
    ]
)

WARMUP_EPOCHS = int(
    training_config[
        "warmup_epochs"
    ]
)

WEIGHT_DECAY = float(
    training_config[
        "weight_decay"
    ]
)

LABEL_SMOOTHING = float(
    training_config[
        "label_smoothing"
    ]
)

DROP_RATE = float(
    training_config[
        "drop_rate"
    ]
)

DROP_PATH_RATE = float(
    training_config[
        "drop_path_rate"
    ]
)

EARLY_STOP_PATIENCE = int(
    training_config[
        "early_stop_patience"
    ]
)

GRAD_CLIP_NORM = float(
    training_config[
        "grad_clip_norm"
    ]
)

USE_MIXED_PRECISION = bool(
    training_config[
        "use_mixed_precision"
    ]
)

USE_TTA = bool(
    training_config[
        "use_tta"
    ]
)

TTA_HORIZONTAL_FLIP = bool(
    training_config[
        "tta_horizontal_flip"
    ]
)

SEEDS = [
    42,
    2025,
    2026,
]

BOOTSTRAP_REPETITIONS = 5000
BOOTSTRAP_SEED = 20260807

EPOCH_CHECKPOINT_INTERVAL = 1

RESUME_COMPLETED_FOLDS = True
RESUME_INTERRUPTED_FOLDS = True
SKIP_COMPLETED_SEED_EXPERIMENTS = True


# ============================================================
# 8. Predeclared factorial ablation configurations
# ============================================================

ABLATION_CONFIGS = {
    "A0_full_locked": {
        "display_name": (
            "Weighted CE + ordinal distance"
        ),

        "role": (
            "locked_primary_model"
        ),

        "use_class_weights": True,

        "use_ordinal": True,

        "ordinal_alpha": 1.0,

        "ordinal_loss_weight": 0.10,

        "train_in_notebook07": False,
    },

    "A1_weighted_ce_only": {
        "display_name": (
            "Weighted CE only"
        ),

        "role": (
            "ablation"
        ),

        "use_class_weights": True,

        "use_ordinal": False,

        "ordinal_alpha": 1.0,

        "ordinal_loss_weight": 0.0,

        "train_in_notebook07": True,
    },

    "A2_unweighted_ce_plus_ordinal": {
        "display_name": (
            "Unweighted CE + ordinal distance"
        ),

        "role": (
            "ablation"
        ),

        "use_class_weights": False,

        "use_ordinal": True,

        "ordinal_alpha": 1.0,

        "ordinal_loss_weight": 0.10,

        "train_in_notebook07": True,
    },

    "A3_unweighted_ce_only": {
        "display_name": (
            "Unweighted CE only"
        ),

        "role": (
            "ablation"
        ),

        "use_class_weights": False,

        "use_ordinal": False,

        "ordinal_alpha": 1.0,

        "ordinal_loss_weight": 0.0,

        "train_in_notebook07": True,
    },
}

ABLATIONS_TO_TRAIN = [
    "A1_weighted_ce_only",
    "A2_unweighted_ce_plus_ordinal",
    "A3_unweighted_ce_only",
]

# Predeclared explanatory contrasts.
CONTRASTS = {
    "ordinal_effect_weighted": {
        "condition_a": (
            "A0_full_locked"
        ),

        "condition_b": (
            "A1_weighted_ce_only"
        ),

        "interpretation": (
            "Effect of adding ordinal loss while "
            "class-weighted CE is retained"
        ),
    },

    "ordinal_effect_unweighted": {
        "condition_a": (
            "A2_unweighted_ce_plus_ordinal"
        ),

        "condition_b": (
            "A3_unweighted_ce_only"
        ),

        "interpretation": (
            "Effect of adding ordinal loss without "
            "class weighting"
        ),
    },

    "class_weight_effect_with_ordinal": {
        "condition_a": (
            "A0_full_locked"
        ),

        "condition_b": (
            "A2_unweighted_ce_plus_ordinal"
        ),

        "interpretation": (
            "Effect of class weighting while "
            "ordinal loss is retained"
        ),
    },

    "class_weight_effect_without_ordinal": {
        "condition_a": (
            "A1_weighted_ce_only"
        ),

        "condition_b": (
            "A3_unweighted_ce_only"
        ),

        "interpretation": (
            "Effect of class weighting when "
            "ordinal loss is absent"
        ),
    },

    "full_vs_plain_ce": {
        "condition_a": (
            "A0_full_locked"
        ),

        "condition_b": (
            "A3_unweighted_ce_only"
        ),

        "interpretation": (
            "Locked full model versus plain "
            "unweighted cross-entropy"
        ),
    },
}

print(
    "\nPredeclared ablation conditions:"
)

for condition_name, config in (
    ABLATION_CONFIGS.items()
):
    print(
        " -",
        condition_name,
        ":",
        config[
            "display_name"
        ],
        "| role =",
        config[
            "role"
        ],
    )


# ============================================================
# 9. Verify full-model OOF sources from Notebook 06
# ============================================================

full_model_source_records = []

for seed in SEEDS:
    image_prediction_path = (
        NOTEBOOK06_ROOT
        / "Swin"
        / f"seed_{seed}"
        / "oof_image_predictions.csv"
    )

    patient_prediction_path = (
        NOTEBOOK06_ROOT
        / "Swin"
        / f"seed_{seed}"
        / "oof_patient_predictions.csv"
    )

    seed_result_path = (
        NOTEBOOK06_ROOT
        / "Swin"
        / f"seed_{seed}"
        / "seed_result.json"
    )

    for path in [
        image_prediction_path,
        patient_prediction_path,
        seed_result_path,
    ]:
        if not path.exists():
            raise FileNotFoundError(
                "Required full-model Notebook 06 "
                f"source is missing:\n{path}"
            )

    full_model_source_records.append({
        "condition_name": (
            "A0_full_locked"
        ),

        "training_seed": int(
            seed
        ),

        "oof_image_prediction_path": str(
            image_prediction_path
        ),

        "oof_image_prediction_sha256": (
            file_sha256(
                image_prediction_path
            )
        ),

        "oof_patient_prediction_path": str(
            patient_prediction_path
        ),

        "oof_patient_prediction_sha256": (
            file_sha256(
                patient_prediction_path
            )
        ),

        "seed_result_path": str(
            seed_result_path
        ),

        "seed_result_sha256": (
            file_sha256(
                seed_result_path
            )
        ),
    })

full_model_source_df = pd.DataFrame(
    full_model_source_records
)

atomic_csv_save(
    full_model_source_df,

    OUTPUT_ROOT
    / "A0_full_model_source_manifest.csv",

    index=False,
)


# ============================================================
# 10. Freeze ablation analysis plan
# ============================================================

SCIENTIFIC_TRAINING_CONFIG = {
    "model_name": (
        SWIN_MODEL_NAME
    ),

    "num_classes": (
        NUM_CLASSES
    ),

    "num_folds": (
        NUM_FOLDS
    ),

    "image_size": (
        IMAGE_SIZE
    ),

    "train_batch_size": (
        TRAIN_BATCH_SIZE
    ),

    "gradient_accumulation_steps": (
        GRADIENT_ACCUMULATION_STEPS
    ),

    "effective_train_batch_size": (
        EFFECTIVE_TRAIN_BATCH_SIZE
    ),

    "validation_batch_size": (
        VALIDATION_BATCH_SIZE
    ),

    "runtime_num_workers": (
        RUNTIME_NUM_WORKERS
    ),

    "epochs": (
        EPOCHS
    ),

    "learning_rate": (
        LEARNING_RATE
    ),

    "min_lr_ratio": (
        MIN_LR_RATIO
    ),

    "warmup_epochs": (
        WARMUP_EPOCHS
    ),

    "weight_decay": (
        WEIGHT_DECAY
    ),

    "label_smoothing": (
        LABEL_SMOOTHING
    ),

    "drop_rate": (
        DROP_RATE
    ),

    "drop_path_rate": (
        DROP_PATH_RATE
    ),

    "early_stop_patience": (
        EARLY_STOP_PATIENCE
    ),

    "grad_clip_norm": (
        GRAD_CLIP_NORM
    ),

    "use_mixed_precision": (
        USE_MIXED_PRECISION
    ),

    "use_tta": (
        USE_TTA
    ),

    "tta_horizontal_flip": (
        TTA_HORIZONTAL_FLIP
    ),

    "primary_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "patient_aggregation": (
        LOCKED_AGGREGATION_RULE
    ),
}

immutable_analysis_plan = {
    "protocol_version": (
        PROTOCOL_VERSION
    ),

    "study_stage": (
        "Explanatory development-only ablation "
        "of the already locked Swin model"
    ),

    "official_test_loaded": False,

    "official_test_evaluated": False,

    "locked_primary_model_condition": (
        "A0_full_locked"
    ),

    "locked_primary_model_can_change": (
        False
    ),

    "ablation_results_used_for_model_selection": (
        False
    ),

    "backbone": (
        "Swin"
    ),

    "model_name": (
        SWIN_MODEL_NAME
    ),

    "ablation_conditions": (
        ABLATION_CONFIGS
    ),

    "predeclared_contrasts": (
        CONTRASTS
    ),

    "training_seeds": (
        SEEDS
    ),

    "training_config": (
        SCIENTIFIC_TRAINING_CONFIG
    ),

    "checkpoint_selection_hierarchy": [
        "Validation patient QWK: higher",
        "Validation patient MAE: lower",
        "Validation patient macro F1: higher",
        "Validation image QWK: higher",
    ],

    "primary_ablation_outcome": (
        "Pooled OOF patient QWK"
    ),

    "secondary_outcomes": [
        "Patient MAE",
        "Patient macro F1",
        "Patient accuracy",
        "Image QWK",
        "Image MAE",
    ],

    "full_model_source_records": (
        full_model_source_records
    ),

    "notebook06_plan_sha256": (
        file_sha256(
            NOTEBOOK06_PLAN_PATH
        )
    ),

    "locked_backbone_sha256": (
        file_sha256(
            LOCKED_BACKBONE_PATH
        )
    ),

    "locked_aggregation_sha256": (
        file_sha256(
            LOCKED_AGGREGATION_PATH
        )
    ),

    "development_manifest_sha256": (
        file_sha256(
            MANIFEST_PATH
        )
    ),

    "fold_assignment_sha256": (
        file_sha256(
            FOLD_ASSIGNMENT_PATH
        )
    ),

    "bootstrap_repetitions": (
        BOOTSTRAP_REPETITIONS
    ),

    "bootstrap_seed": (
        BOOTSTRAP_SEED
    ),

    "interpretation_constraint": (
        "The full model was locked before this "
        "ablation. No ablation result may replace "
        "or modify the primary model."
    ),
}

analysis_plan_fingerprint = (
    dictionary_sha256(
        immutable_analysis_plan
    )
)

analysis_plan = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    **immutable_analysis_plan,
}

analysis_plan_path = (
    OUTPUT_ROOT
    / "analysis_plan_frozen_before_ablation_training.json"
)

if analysis_plan_path.exists():
    with open(
        analysis_plan_path,
        "r",
        encoding="utf-8",
    ) as file:
        existing_plan = json.load(
            file
        )

    if (
        existing_plan.get(
            "analysis_plan_fingerprint"
        )
        !=
        analysis_plan_fingerprint
    ):
        raise RuntimeError(
            "Existing Notebook 07 analysis plan "
            "differs from this configuration.\n"
            "Do not mix protocols. Use a new "
            "versioned OUTPUT_ROOT."
        )

    print(
        "\nExisting ablation plan validated:",
        analysis_plan_path,
    )

else:
    atomic_json_save(
        analysis_plan,
        analysis_plan_path,
    )

    print(
        "\nSaved frozen ablation plan:",
        analysis_plan_path,
    )

print(
    "Ablation-plan fingerprint:",
    analysis_plan_fingerprint,
)


# ============================================================
# 11. Environment metadata
# ============================================================

environment_metadata = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "python": (
        platform.python_version()
    ),

    "platform": (
        platform.platform()
    ),

    "torch": (
        torch.__version__
    ),

    "torchvision": (
        torchvision.__version__
    ),

    "timm": (
        timm.__version__
    ),

    "numpy": (
        np.__version__
    ),

    "pandas": (
        pd.__version__
    ),

    "scikit_learn": (
        safe_package_version(
            "scikit-learn"
        )
    ),

    "pillow": (
        safe_package_version(
            "Pillow"
        )
    ),

    "cuda_available": bool(
        torch.cuda.is_available()
    ),

    "cuda_version": (
        torch.version.cuda
    ),

    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "official_test_loaded": False,

    "official_test_evaluated": False,
}

atomic_json_save(
    environment_metadata,

    OUTPUT_ROOT
    / "environment_metadata.json",
)


# ============================================================
# 12. Verify timm Swin identifier
# ============================================================

base_model_name = (
    SWIN_MODEL_NAME
    .split(
        ".",
        1,
    )[0]
)

if (
    base_model_name
    not in
    set(
        timm.list_models()
    )
):
    raise RuntimeError(
        "Frozen Swin base model is unavailable:\n"
        f"{base_model_name}"
    )

try:
    verification_model = (
        timm.create_model(
            SWIN_MODEL_NAME,

            pretrained=False,

            num_classes=(
                NUM_CLASSES
            ),

            drop_rate=(
                DROP_RATE
            ),

            drop_path_rate=(
                DROP_PATH_RATE
            ),
        )
    )

    del verification_model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as error:
    raise RuntimeError(
        "Frozen Swin model identifier could not "
        "be instantiated:\n"
        f"{SWIN_MODEL_NAME}\n"
        f"{repr(error)}"
    ) from error

print(
    "\nVerified Swin timm model:",
    SWIN_MODEL_NAME,
)


# ============================================================
# 13. Load and validate development manifest
# ============================================================

trainval_df = pd.read_csv(
    MANIFEST_PATH
)

required_manifest_columns = {
    "relative_path",
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
}

missing_columns = (
    required_manifest_columns
    -
    set(
        trainval_df.columns
    )
)

if missing_columns:
    raise RuntimeError(
        "Development manifest is missing columns: "
        f"{sorted(missing_columns)}"
    )

trainval_df["path"] = (
    trainval_df[
        "relative_path"
    ]
    .map(
        lambda relative_path: str(
            DEVELOPMENT_ROOT
            /
            Path(relative_path)
        )
    )
)

for column in [
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
]:
    trainval_df[
        column
    ] = (
        trainval_df[
            column
        ]
        .astype(int)
    )

trainval_df = (
    trainval_df
    .sort_values(
        [
            "patient_id",
            "relative_path",
        ]
    )
    .reset_index(
        drop=True
    )
)

if len(trainval_df) != 9590:
    raise RuntimeError(
        "Expected 9,590 development images; "
        f"found {len(trainval_df)}."
    )

if (
    trainval_df[
        "patient_id"
    ].nunique()
    !=
    478
):
    raise RuntimeError(
        "Expected 478 development patients."
    )

if (
    trainval_df[
        "path"
    ].duplicated().any()
):
    raise RuntimeError(
        "Duplicate image paths exist."
    )

if (
    set(
        trainval_df[
            "cv_fold"
        ].unique()
    )
    !=
    set(
        range(
            NUM_FOLDS
        )
    )
):
    raise RuntimeError(
        "Frozen fold IDs are invalid."
    )


# ============================================================
# 14. Validate frozen reference
# ============================================================

reference_audit_df = pd.read_csv(
    REFERENCE_AUDIT_PATH
)

reference_audit_df[
    "patient_id"
] = (
    reference_audit_df[
        "patient_id"
    ]
    .astype(int)
)

reference_audit_df[
    PRIMARY_REFERENCE_RULE
] = (
    reference_audit_df[
        PRIMARY_REFERENCE_RULE
    ]
    .astype(int)
)

manifest_patient_df = (
    trainval_df[
        [
            "patient_id",
            "patient_y_true",
            "cv_fold",
        ]
    ]
    .drop_duplicates()
)

reference_check_df = (
    manifest_patient_df
    .merge(
        reference_audit_df[
            [
                "patient_id",
                PRIMARY_REFERENCE_RULE,
            ]
        ],

        on="patient_id",

        how="inner",

        validate="one_to_one",
    )
)

if len(reference_check_df) != 478:
    raise RuntimeError(
        "Patient-reference alignment failure."
    )

if not (
    reference_check_df[
        "patient_y_true"
    ]
    ==
    reference_check_df[
        PRIMARY_REFERENCE_RULE
    ]
).all():
    raise RuntimeError(
        "Manifest patient reference differs "
        "from modal_low."
    )

print(
    "\nDevelopment images:",
    len(trainval_df),
)

print(
    "Development patients:",
    trainval_df[
        "patient_id"
    ].nunique(),
)

print(
    "Fold patient counts:",
    manifest_patient_df[
        "cv_fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict(),
)


# ============================================================
# 15. Runtime image cache
# ============================================================

USE_LOCAL_IMAGE_CACHE = True

LOCAL_IMAGE_CACHE_ROOT = Path(
    "/content/LIMUC_dev_cache_ablation07"
)

LOCAL_IMAGE_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

IMAGE_IO_MAX_RETRIES = 12

IMAGE_IO_INITIAL_DELAY_SECONDS = 1.0

IMAGE_IO_MAX_DELAY_SECONDS = 8.0


def cache_path_for_relative_path(
    relative_path,
):
    return (
        LOCAL_IMAGE_CACHE_ROOT
        /
        Path(
            str(relative_path)
        )
    )


def read_drive_bytes(
    source_path,
):
    source_path = str(
        source_path
    )

    last_error = None

    for attempt_index in range(
        IMAGE_IO_MAX_RETRIES
    ):
        try:
            with open(
                source_path,
                "rb",
            ) as file:
                data = file.read()

            if len(data) == 0:
                raise OSError(
                    "Zero-byte image read."
                )

            return data

        except (
            OSError,
            IOError,
        ) as error:
            last_error = error

            if (
                attempt_index
                ==
                IMAGE_IO_MAX_RETRIES - 1
            ):
                break

            delay = retry_delay(
                attempt_index,
                IMAGE_IO_INITIAL_DELAY_SECONDS,
                IMAGE_IO_MAX_DELAY_SECONDS,
            )

            print(
                "\nTransient Drive read failure:"
            )

            print(
                source_path
            )

            print(
                "Error:",
                repr(error),
            )

            print(
                f"Retrying in {delay:.1f}s..."
            )

            time.sleep(
                delay
            )

    raise OSError(
        "Could not read image:\n"
        f"{source_path}\n"
        f"Last error: {repr(last_error)}"
    )


def ensure_cached_image(
    source_path,
    relative_path,
):
    local_path = (
        cache_path_for_relative_path(
            relative_path
        )
    )

    if (
        local_path.exists()
        and
        local_path.stat().st_size > 0
    ):
        return local_path

    local_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    image_bytes = (
        read_drive_bytes(
            source_path
        )
    )

    temp_path = temporary_path(
        local_path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "wb",
    ) as file:
        file.write(
            image_bytes
        )

    os.replace(
        temp_path,
        local_path,
    )

    return local_path


def load_rgb_image(
    source_path,
    relative_path,
):
    local_path = None
    last_error = None

    for attempt_index in range(
        IMAGE_IO_MAX_RETRIES
    ):
        try:
            if USE_LOCAL_IMAGE_CACHE:
                local_path = (
                    ensure_cached_image(
                        source_path,
                        relative_path,
                    )
                )

                read_path = (
                    local_path
                )

            else:
                read_path = Path(
                    source_path
                )

            with open(
                read_path,
                "rb",
            ) as file:
                image_bytes = (
                    file.read()
                )

            if len(image_bytes) == 0:
                raise OSError(
                    "Zero-byte cached image."
                )

            with Image.open(
                io.BytesIO(
                    image_bytes
                )
            ) as image:
                image.load()

                return image.convert(
                    "RGB"
                )

        except (
            OSError,
            IOError,
            ValueError,
        ) as error:
            last_error = error

            if (
                local_path is not None
                and
                local_path.exists()
            ):
                try:
                    local_path.unlink()
                except Exception:
                    pass

            if (
                attempt_index
                ==
                IMAGE_IO_MAX_RETRIES - 1
            ):
                break

            delay = retry_delay(
                attempt_index,
                IMAGE_IO_INITIAL_DELAY_SECONDS,
                IMAGE_IO_MAX_DELAY_SECONDS,
            )

            time.sleep(
                delay
            )

    raise OSError(
        "Could not decode image:\n"
        f"{source_path}\n"
        f"Last error: {repr(last_error)}"
    )


# ============================================================
# 16. Dataset and transforms
# ============================================================

class LIMUCDataset(
    Dataset
):
    def __init__(
        self,
        dataframe,
        transform=None,
    ):
        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.transform = (
            transform
        )

    def __len__(
        self,
    ):
        return len(
            self.df
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.df.iloc[
            index
        ]

        image = load_rgb_image(
            row[
                "path"
            ],

            row[
                "relative_path"
            ],
        )

        if (
            self.transform
            is not None
        ):
            image = self.transform(
                image
            )

        return {
            "image": (
                image
            ),

            "label": torch.tensor(
                int(
                    row[
                        "label"
                    ]
                ),

                dtype=torch.long,
            ),

            "patient_id": torch.tensor(
                int(
                    row[
                        "patient_id"
                    ]
                ),

                dtype=torch.long,
            ),

            "patient_y_true": torch.tensor(
                int(
                    row[
                        "patient_y_true"
                    ]
                ),

                dtype=torch.long,
            ),

            "path": str(
                row[
                    "path"
                ]
            ),
        }


def build_train_transform():
    return transforms.Compose([
        transforms.RandomResizedCrop(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),

            scale=(
                0.75,
                1.00,
            ),

            ratio=(
                0.90,
                1.10,
            ),
        ),

        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        transforms.RandomRotation(
            degrees=8
        ),

        transforms.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.08,
            hue=0.02,
        ),

        transforms.GaussianBlur(
            kernel_size=3,

            sigma=(
                0.1,
                1.2,
            ),
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],

            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


def build_eval_transform():
    return transforms.Compose([
        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],

            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


# ============================================================
# 17. Metrics and patient aggregation
# ============================================================

PROBABILITY_COLUMNS = [
    "prob_0",
    "prob_1",
    "prob_2",
    "prob_3",
]


def compute_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    if len(y_true) == 0:
        raise ValueError(
            "Empty metric input."
        )

    qwk = cohen_kappa_score(
        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
        ],

        weights="quadratic",
    )

    return {
        "qwk": (
            float(qwk)
            if np.isfinite(qwk)
            else np.nan
        ),

        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),

        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,

                labels=[
                    0,
                    1,
                    2,
                    3,
                ],

                average="macro",

                zero_division=0,
            )
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
    }


def make_confusion_df(
    y_true,
    y_pred,
):
    return pd.DataFrame(
        confusion_matrix(
            y_true,
            y_pred,

            labels=[
                0,
                1,
                2,
                3,
            ],
        ),

        index=[
            "true_0",
            "true_1",
            "true_2",
            "true_3",
        ],

        columns=[
            "pred_0",
            "pred_1",
            "pred_2",
            "pred_3",
        ],
    )


def aggregate_patient_probabilities(
    image_df,
):
    patient_label_count = (
        image_df
        .groupby(
            "patient_id"
        )[
            "patient_y_true"
        ]
        .nunique()
    )

    if (
        patient_label_count.max()
        != 1
    ):
        raise RuntimeError(
            "Inconsistent patient reference."
        )

    probability_df = (
        image_df
        .groupby(
            "patient_id",
            as_index=False,
        )[
            PROBABILITY_COLUMNS
        ]
        .mean()
    )

    metadata_df = (
        image_df
        .groupby(
            "patient_id",
            as_index=False,
        )
        .agg(
            y_true=(
                "patient_y_true",
                "first",
            ),

            n_images=(
                "path",
                "size",
            ),
        )
    )

    patient_df = (
        metadata_df
        .merge(
            probability_df,

            on="patient_id",

            how="inner",

            validate="one_to_one",
        )
    )

    patient_df[
        "patient_id"
    ] = (
        patient_df[
            "patient_id"
        ]
        .astype(int)
    )

    patient_df[
        "y_true"
    ] = (
        patient_df[
            "y_true"
        ]
        .astype(int)
    )

    patient_df[
        "y_pred"
    ] = (
        patient_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy()
        .argmax(
            axis=1
        )
        .astype(int)
    )

    return (
        patient_df
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 18. Validate prediction files
# ============================================================

def validate_image_predictions(
    prediction_df,
    expected_df,
    expected_seed=None,
    expected_fold=None,
):
    prediction_df = (
        prediction_df
        .copy()
    )

    required_columns = {
        "path",
        "patient_id",
        "y_true",
        "patient_y_true",
        "y_pred",
        *PROBABILITY_COLUMNS,
    }

    missing_columns = (
        required_columns
        -
        set(
            prediction_df.columns
        )
    )

    if missing_columns:
        raise RuntimeError(
            "Prediction file missing columns: "
            f"{sorted(missing_columns)}"
        )

    prediction_df[
        "path"
    ] = (
        prediction_df[
            "path"
        ].astype(str)
    )

    for column in [
        "patient_id",
        "y_true",
        "patient_y_true",
        "y_pred",
    ]:
        prediction_df[
            column
        ] = (
            prediction_df[
                column
            ]
            .astype(int)
        )

    if (
        prediction_df[
            "path"
        ].duplicated().any()
    ):
        raise RuntimeError(
            "Duplicate prediction paths."
        )

    if (
        len(
            prediction_df
        )
        !=
        len(
            expected_df
        )
    ):
        raise RuntimeError(
            "Prediction row count differs "
            "from expected data."
        )

    probabilities = (
        prediction_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )

    if not np.isfinite(
        probabilities
    ).all():
        raise RuntimeError(
            "Non-finite probabilities."
        )

    if not np.allclose(
        probabilities.sum(
            axis=1
        ),

        1.0,

        atol=2e-3,

        rtol=0.0,
    ):
        raise RuntimeError(
            "Probability rows do not sum to one."
        )

    prediction_df[
        "y_pred"
    ] = (
        probabilities
        .argmax(
            axis=1
        )
        .astype(int)
    )

    expected_metadata = (
        expected_df[
            [
                "path",
                "patient_id",
                "label",
                "patient_y_true",
            ]
        ]
        .rename(
            columns={
                "patient_id": (
                    "patient_id_expected"
                ),

                "label": (
                    "y_true_expected"
                ),

                "patient_y_true": (
                    "patient_y_true_expected"
                ),
            }
        )
    )

    check_df = (
        prediction_df[
            [
                "path",
                "patient_id",
                "y_true",
                "patient_y_true",
            ]
        ]
        .merge(
            expected_metadata,

            on="path",

            how="inner",

            validate="one_to_one",
        )
    )

    if len(check_df) != len(
        expected_df
    ):
        raise RuntimeError(
            "Prediction paths do not align "
            "with expected data."
        )

    if not (
        check_df[
            "patient_id"
        ]
        ==
        check_df[
            "patient_id_expected"
        ]
    ).all():
        raise RuntimeError(
            "Patient ID mismatch."
        )

    if not (
        check_df[
            "y_true"
        ]
        ==
        check_df[
            "y_true_expected"
        ]
    ).all():
        raise RuntimeError(
            "Image-label mismatch."
        )

    if not (
        check_df[
            "patient_y_true"
        ]
        ==
        check_df[
            "patient_y_true_expected"
        ]
    ).all():
        raise RuntimeError(
            "Patient-reference mismatch."
        )

    if (
        expected_seed
        is not None
    ):
        prediction_df[
            "training_seed"
        ] = int(
            expected_seed
        )

    if (
        expected_fold
        is not None
    ):
        prediction_df[
            "fold"
        ] = int(
            expected_fold
        )

    return (
        prediction_df
        .sort_values(
            "path"
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 19. Loss functions
# ============================================================

class ExpectedOrdinalDistanceLoss(
    nn.Module
):
    def __init__(
        self,
        num_classes,
        alpha,
    ):
        super().__init__()

        self.alpha = float(
            alpha
        )

        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )

    def forward(
        self,
        logits,
        targets,
    ):
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        target_values = (
            targets
            .float()
            .unsqueeze(1)
        )

        distances = torch.abs(
            target_values
            -
            self.class_values.unsqueeze(
                0
            )
        ) ** self.alpha

        return (
            probabilities
            *
            distances
        ).sum(
            dim=1
        ).mean()


class AblationLoss(
    nn.Module
):
    def __init__(
        self,
        ce_weight,
        use_ordinal,
        ordinal_alpha,
        ordinal_loss_weight,
    ):
        super().__init__()

        self.use_ordinal = bool(
            use_ordinal
        )

        self.ordinal_loss_weight = float(
            ordinal_loss_weight
        )

        self.cross_entropy = (
            nn.CrossEntropyLoss(
                weight=(
                    ce_weight
                ),

                label_smoothing=(
                    LABEL_SMOOTHING
                ),
            )
        )

        if self.use_ordinal:
            self.ordinal_loss = (
                ExpectedOrdinalDistanceLoss(
                    NUM_CLASSES,

                    ordinal_alpha,
                )
            )

        else:
            self.ordinal_loss = None

    def forward(
        self,
        logits,
        targets,
    ):
        ce_loss = (
            self.cross_entropy(
                logits,
                targets,
            )
        )

        if self.ordinal_loss is None:
            ordinal_loss = torch.zeros(
                (),
                dtype=logits.dtype,
                device=logits.device,
            )

            total_loss = (
                ce_loss
            )

        else:
            ordinal_loss = (
                self.ordinal_loss(
                    logits,
                    targets,
                )
            )

            total_loss = (
                ce_loss
                +
                self.ordinal_loss_weight
                *
                ordinal_loss
            )

        return (
            total_loss,
            ce_loss.detach(),
            ordinal_loss.detach(),
        )


# ============================================================
# 20. RNG / resume helpers
# ============================================================

def seed_everything(
    seed,
):
    os.environ[
        "PYTHONHASHSEED"
    ] = str(
        seed
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed(
            seed
        )

        torch.cuda.manual_seed_all(
            seed
        )


def seed_worker(
    worker_id,
):
    worker_seed = (
        torch.initial_seed()
        %
        (2**32)
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


def as_cpu_byte_tensor(
    state,
    name,
):
    if isinstance(
        state,
        torch.Tensor,
    ):
        return (
            state
            .detach()
            .to(
                "cpu",
                dtype=torch.uint8,
            )
            .contiguous()
        )

    if isinstance(
        state,
        np.ndarray,
    ):
        return (
            torch.from_numpy(
                state.astype(
                    np.uint8,
                    copy=False,
                )
            )
            .cpu()
            .contiguous()
        )

    return torch.tensor(
        state,
        dtype=torch.uint8,
        device="cpu",
    ).contiguous()


def capture_rng_state():
    state = {
        "python": (
            random.getstate()
        ),

        "numpy": (
            np.random.get_state()
        ),

        "torch_cpu": (
            torch.get_rng_state()
            .cpu()
            .contiguous()
        ),

        "torch_cuda": None,
    }

    if torch.cuda.is_available():
        state[
            "torch_cuda"
        ] = [
            value
            .cpu()
            .contiguous()

            for value in (
                torch.cuda
                .get_rng_state_all()
            )
        ]

    return state


def restore_rng_state(
    state,
):
    random.setstate(
        state[
            "python"
        ]
    )

    numpy_state = (
        state[
            "numpy"
        ]
    )

    if isinstance(
        numpy_state,
        list,
    ):
        numpy_state = tuple(
            numpy_state
        )

    np.random.set_state(
        numpy_state
    )

    torch.set_rng_state(
        as_cpu_byte_tensor(
            state[
                "torch_cpu"
            ],

            "CPU RNG",
        )
    )

    cuda_states = (
        state.get(
            "torch_cuda"
        )
    )

    if (
        torch.cuda.is_available()
        and
        cuda_states is not None
    ):
        normalized = [
            as_cpu_byte_tensor(
                value,

                f"CUDA RNG {index}",
            )

            for index, value in enumerate(
                cuda_states
            )
        ]

        if (
            len(normalized)
            !=
            torch.cuda.device_count()
        ):
            if (
                torch.cuda.device_count()
                ==
                1
                and
                len(normalized)
                >=
                1
            ):
                normalized = [
                    normalized[
                        0
                    ]
                ]

            else:
                raise RuntimeError(
                    "CUDA RNG count mismatch."
                )

        torch.cuda.set_rng_state_all(
            normalized
        )


def optimizer_to_device(
    optimizer,
    target_device,
):
    for optimizer_state in (
        optimizer.state.values()
    ):
        for key, value in list(
            optimizer_state.items()
        ):
            if isinstance(
                value,
                torch.Tensor,
            ):
                optimizer_state[
                    key
                ] = value.to(
                    target_device
                )


seed_everything(
    SEEDS[
        0
    ]
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.set_float32_matmul_precision(
        "high"
    )
except Exception:
    pass


# ============================================================
# 21. Scheduler / AMP
# ============================================================

def build_scheduler(
    optimizer,
):
    def lr_lambda(
        epoch,
    ):
        if epoch < WARMUP_EPOCHS:
            return (
                float(
                    epoch + 1
                )
                /
                float(
                    max(
                        1,
                        WARMUP_EPOCHS,
                    )
                )
            )

        progress = (
            float(
                epoch
                -
                WARMUP_EPOCHS
            )
            /
            float(
                max(
                    1,
                    EPOCHS
                    -
                    WARMUP_EPOCHS,
                )
            )
        )

        cosine = (
            0.5
            *
            (
                1.0
                +
                math.cos(
                    math.pi
                    *
                    progress
                )
            )
        )

        return max(
            MIN_LR_RATIO,
            cosine,
        )

    return (
        torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda,
        )
    )


def create_scaler():
    try:
        return torch.amp.GradScaler(
            "cuda",

            enabled=(
                USE_MIXED_PRECISION
                and
                torch.cuda.is_available()
            ),
        )

    except Exception:
        return torch.cuda.amp.GradScaler(
            enabled=(
                USE_MIXED_PRECISION
                and
                torch.cuda.is_available()
            )
        )


def amp_context(
    enabled,
):
    try:
        return torch.amp.autocast(
            device_type="cuda",
            enabled=enabled,
        )

    except Exception:
        return torch.cuda.amp.autocast(
            enabled=enabled
        )


# ============================================================
# 22. Class weights
# ============================================================

def calculate_class_weights(
    dataframe,
    device,
):
    counts = (
        dataframe[
            "label"
        ]
        .value_counts()
        .reindex(
            range(
                NUM_CLASSES
            ),

            fill_value=0,
        )
        .sort_index()
    )

    if (
        counts == 0
    ).any():
        raise RuntimeError(
            "Training fold is missing a class."
        )

    weights = (
        counts.sum()
        /
        (
            NUM_CLASSES
            *
            counts
        )
    )

    return (
        torch.tensor(
            weights.to_numpy(),

            dtype=torch.float32,

            device=device,
        ),

        {
            str(key): int(
                value
            )

            for key, value in (
                counts.items()
            )
        },

        {
            str(key): float(
                value
            )

            for key, value in (
                weights.items()
            )
        },
    )


# ============================================================
# 23. Train and evaluate
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
):
    model.train()

    total_loss = 0.0

    y_true = []
    y_pred = []

    use_amp = (
        USE_MIXED_PRECISION
        and
        device.type
        ==
        "cuda"
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    number_of_batches = len(
        loader
    )

    for batch_index, batch in enumerate(
        loader
    ):
        images = (
            batch[
                "image"
            ]
            .to(
                device,
                non_blocking=True,
            )
        )

        labels = (
            batch[
                "label"
            ]
            .to(
                device,
                non_blocking=True,
            )
        )

        group_start = (
            batch_index
            //
            GRADIENT_ACCUMULATION_STEPS
        ) * GRADIENT_ACCUMULATION_STEPS

        group_size = min(
            GRADIENT_ACCUMULATION_STEPS,

            number_of_batches
            -
            group_start,
        )

        with amp_context(
            use_amp
        ):
            logits = model(
                images
            )

            (
                raw_loss,
                _,
                _,
            ) = criterion(
                logits,
                labels,
            )

            scaled_loss = (
                raw_loss
                /
                group_size
            )

        if use_amp:
            scaler.scale(
                scaled_loss
            ).backward()

        else:
            scaled_loss.backward()

        should_step = (
            (
                batch_index + 1
            )
            %
            GRADIENT_ACCUMULATION_STEPS
            ==
            0
            or
            (
                batch_index + 1
            )
            ==
            number_of_batches
        )

        if should_step:
            if use_amp:
                scaler.unscale_(
                    optimizer
                )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),

                GRAD_CLIP_NORM,
            )

            if use_amp:
                scaler.step(
                    optimizer
                )

                scaler.update()

            else:
                optimizer.step()

            optimizer.zero_grad(
                set_to_none=True
            )

        total_loss += (
            float(
                raw_loss.item()
            )
            *
            images.size(
                0
            )
        )

        predictions = (
            logits.argmax(
                dim=1
            )
        )

        y_true.extend(
            labels.detach()
            .cpu()
            .tolist()
        )

        y_pred.extend(
            predictions.detach()
            .cpu()
            .tolist()
        )

    metrics = compute_metrics(
        y_true,
        y_pred,
    )

    metrics[
        "loss"
    ] = (
        total_loss
        /
        len(
            loader.dataset
        )
    )

    return metrics


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
):
    model.eval()

    total_loss = 0.0

    image_truth = []

    patient_truth = []

    patient_ids = []

    paths = []

    probability_rows = []

    use_amp = (
        USE_MIXED_PRECISION
        and
        device.type
        ==
        "cuda"
    )

    for batch in loader:
        images = (
            batch[
                "image"
            ]
            .to(
                device,
                non_blocking=True,
            )
        )

        labels = (
            batch[
                "label"
            ]
            .to(
                device,
                non_blocking=True,
            )
        )

        with amp_context(
            use_amp
        ):
            logits = model(
                images
            )

            (
                loss,
                _,
                _,
            ) = criterion(
                logits,
                labels,
            )

            probabilities_list = [
                torch.softmax(
                    logits,
                    dim=1,
                )
            ]

            if (
                USE_TTA
                and
                TTA_HORIZONTAL_FLIP
            ):
                flipped_logits = (
                    model(
                        torch.flip(
                            images,
                            dims=[
                                3
                            ],
                        )
                    )
                )

                probabilities_list.append(
                    torch.softmax(
                        flipped_logits,
                        dim=1,
                    )
                )

            probabilities = (
                torch.stack(
                    probabilities_list,
                    dim=0,
                )
                .mean(
                    dim=0
                )
            )

        total_loss += (
            float(
                loss.item()
            )
            *
            images.size(
                0
            )
        )

        image_truth.extend(
            labels
            .cpu()
            .tolist()
        )

        patient_truth.extend(
            batch[
                "patient_y_true"
            ]
            .cpu()
            .tolist()
        )

        patient_ids.extend(
            batch[
                "patient_id"
            ]
            .cpu()
            .tolist()
        )

        paths.extend(
            batch[
                "path"
            ]
        )

        probability_rows.extend(
            probabilities
            .cpu()
            .numpy()
            .tolist()
        )

    image_df = pd.DataFrame({
        "path": (
            paths
        ),

        "patient_id": (
            patient_ids
        ),

        "y_true": (
            image_truth
        ),

        "patient_y_true": (
            patient_truth
        ),
    })

    probability_df = pd.DataFrame(
        probability_rows,

        columns=(
            PROBABILITY_COLUMNS
        ),
    )

    image_df = pd.concat(
        [
            image_df,
            probability_df,
        ],

        axis=1,
    )

    image_df[
        "y_pred"
    ] = (
        image_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy()
        .argmax(
            axis=1
        )
        .astype(int)
    )

    image_metrics = compute_metrics(
        image_df[
            "y_true"
        ],

        image_df[
            "y_pred"
        ],
    )

    image_metrics[
        "loss"
    ] = (
        total_loss
        /
        len(
            loader.dataset
        )
    )

    patient_df = (
        aggregate_patient_probabilities(
            image_df
        )
    )

    patient_metrics = (
        compute_metrics(
            patient_df[
                "y_true"
            ],

            patient_df[
                "y_pred"
            ],
        )
    )

    return (
        image_metrics,
        patient_metrics,
        image_df,
        patient_df,
    )


# ============================================================
# 24. Checkpoint hierarchy
# ============================================================

def max_metric(
    value,
):
    if (
        value is None
        or
        not np.isfinite(
            value
        )
    ):
        return -np.inf

    return float(
        value
    )


def min_metric(
    value,
):
    if (
        value is None
        or
        not np.isfinite(
            value
        )
    ):
        return np.inf

    return float(
        value
    )


def checkpoint_is_better(
    patient_metrics,
    best_patient_metrics,
    image_metrics,
    best_image_metrics,
):
    if (
        best_patient_metrics
        is None
    ):
        return True

    candidate_tuple = (
        max_metric(
            patient_metrics[
                "qwk"
            ]
        ),

        -min_metric(
            patient_metrics[
                "mae"
            ]
        ),

        max_metric(
            patient_metrics[
                "macro_f1"
            ]
        ),

        max_metric(
            image_metrics[
                "qwk"
            ]
        ),
    )

    best_tuple = (
        max_metric(
            best_patient_metrics[
                "qwk"
            ]
        ),

        -min_metric(
            best_patient_metrics[
                "mae"
            ]
        ),

        max_metric(
            best_patient_metrics[
                "macro_f1"
            ]
        ),

        max_metric(
            best_image_metrics[
                "qwk"
            ]
        ),
    )

    return (
        candidate_tuple
        >
        best_tuple
    )


# ============================================================
# 25. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "\nDevice:",
    device,
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        ),
    )


# ============================================================
# 26. Run or resume one ablation fold
# ============================================================

def run_ablation_fold(
    condition_name,
    condition_config,
    training_seed,
    fold,
    seed_directory,
):
    fold_seed = (
        int(
            training_seed
        )
        +
        int(
            fold
        )
    )

    fold_directory = (
        seed_directory
        /
        f"fold_{fold}"
    )

    fold_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    fold_configuration = {
        "protocol_version": (
            PROTOCOL_VERSION
        ),

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),

        "condition_name": (
            condition_name
        ),

        "condition_config": (
            condition_config
        ),

        "training_seed": (
            int(
                training_seed
            )
        ),

        "fold": (
            int(
                fold
            )
        ),

        "fold_seed": (
            int(
                fold_seed
            )
        ),
    }

    fold_fingerprint = (
        dictionary_sha256(
            fold_configuration
        )
    )

    training_df = (
        trainval_df[
            trainval_df[
                "cv_fold"
            ]
            !=
            fold
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    validation_df = (
        trainval_df[
            trainval_df[
                "cv_fold"
            ]
            ==
            fold
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    if (
        set(
            training_df[
                "patient_id"
            ].unique()
        )
        &
        set(
            validation_df[
                "patient_id"
            ].unique()
        )
    ):
        raise RuntimeError(
            "Patient leakage detected."
        )

    fold_result_path = (
        fold_directory
        / "fold_result.json"
    )

    image_prediction_path = (
        fold_directory
        / "validation_image_predictions.csv"
    )

    patient_prediction_path = (
        fold_directory
        / "validation_patient_predictions.csv"
    )

    best_checkpoint_path = (
        fold_directory
        / "best_checkpoint.pt"
    )

    last_checkpoint_path = (
        fold_directory
        / "last_epoch_checkpoint.pt"
    )

    history_path = (
        fold_directory
        / "history.csv"
    )

    if (
        RESUME_COMPLETED_FOLDS
        and
        fold_result_path.exists()
        and
        image_prediction_path.exists()
        and
        patient_prediction_path.exists()
        and
        best_checkpoint_path.exists()
    ):
        with open(
            fold_result_path,
            "r",
            encoding="utf-8",
        ) as file:
            fold_result = (
                json.load(
                    file
                )
            )

        if (
            fold_result.get(
                "fold_fingerprint"
            )
            !=
            fold_fingerprint
        ):
            raise RuntimeError(
                "Completed fold fingerprint mismatch."
            )

        image_prediction_df = pd.read_csv(
            image_prediction_path
        )

        image_prediction_df = (
            validate_image_predictions(
                image_prediction_df,

                validation_df,

                expected_seed=(
                    training_seed
                ),

                expected_fold=(
                    fold
                ),
            )
        )

        patient_prediction_df = (
            aggregate_patient_probabilities(
                image_prediction_df
            )
        )

        patient_prediction_df[
            "fold"
        ] = int(
            fold
        )

        patient_prediction_df[
            "training_seed"
        ] = int(
            training_seed
        )

        print(
            "\nLoaded completed fold:",
            condition_name,
            "| seed",
            training_seed,
            "| fold",
            fold,
        )

        return {
            "fold_result": (
                fold_result
            ),

            "image_predictions": (
                image_prediction_df
            ),

            "patient_predictions": (
                patient_prediction_df
            ),
        }

    print(
        "\n"
        +
        "=" * 115
    )

    print(
        f"{condition_name} | "
        f"seed={training_seed} | "
        f"fold={fold + 1}/{NUM_FOLDS}"
    )

    print(
        "=" * 115
    )

    print(
        "Train images:",
        len(
            training_df
        ),
    )

    print(
        "Validation images:",
        len(
            validation_df
        ),
    )

    print(
        "Use class weights:",
        condition_config[
            "use_class_weights"
        ],
    )

    print(
        "Use ordinal loss:",
        condition_config[
            "use_ordinal"
        ],
    )

    training_dataset = (
        LIMUCDataset(
            training_df,

            transform=(
                build_train_transform()
            ),
        )
    )

    validation_dataset = (
        LIMUCDataset(
            validation_df,

            transform=(
                build_eval_transform()
            ),
        )
    )

    training_generator = (
        torch.Generator()
    )

    validation_generator = (
        torch.Generator()
    )

    training_generator.manual_seed(
        fold_seed
    )

    validation_generator.manual_seed(
        fold_seed
        +
        100000
    )

    training_loader = DataLoader(
        training_dataset,

        batch_size=(
            TRAIN_BATCH_SIZE
        ),

        shuffle=True,

        num_workers=(
            RUNTIME_NUM_WORKERS
        ),

        pin_memory=(
            device.type
            ==
            "cuda"
        ),

        drop_last=False,

        worker_init_fn=(
            seed_worker
        ),

        generator=(
            training_generator
        ),

        persistent_workers=False,
    )

    validation_loader = DataLoader(
        validation_dataset,

        batch_size=(
            VALIDATION_BATCH_SIZE
        ),

        shuffle=False,

        num_workers=(
            RUNTIME_NUM_WORKERS
        ),

        pin_memory=(
            device.type
            ==
            "cuda"
        ),

        drop_last=False,

        worker_init_fn=(
            seed_worker
        ),

        generator=(
            validation_generator
        ),

        persistent_workers=False,
    )

    resume_payload = None

    if (
        RESUME_INTERRUPTED_FOLDS
        and
        last_checkpoint_path.exists()
    ):
        try:
            resume_payload = (
                safe_torch_load(
                    last_checkpoint_path,

                    map_location="cpu",
                )
            )

            if (
                resume_payload.get(
                    "fold_fingerprint"
                )
                !=
                fold_fingerprint
            ):
                raise RuntimeError(
                    "Resume checkpoint "
                    "fingerprint mismatch."
                )

            print(
                "\nFound resumable checkpoint:",
                last_checkpoint_path,
            )

        except Exception as error:
            corrupt_path = (
                last_checkpoint_path
                .with_name(
                    "last_epoch_checkpoint_corrupt_"
                    +
                    datetime.now(
                        timezone.utc
                    ).strftime(
                        "%Y%m%d_%H%M%S"
                    )
                    +
                    ".pt"
                )
            )

            os.replace(
                last_checkpoint_path,
                corrupt_path,
            )

            print(
                "Corrupt/incompatible resume "
                "checkpoint moved to:",
                corrupt_path,
            )

            print(
                "Fold will restart."
            )

            print(
                "Error:",
                repr(error),
            )

            resume_payload = None

    seed_everything(
        fold_seed
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = timm.create_model(
        SWIN_MODEL_NAME,

        pretrained=(
            resume_payload
            is None
        ),

        num_classes=(
            NUM_CLASSES
        ),

        drop_rate=(
            DROP_RATE
        ),

        drop_path_rate=(
            DROP_PATH_RATE
        ),
    ).to(
        device
    )

    parameter_count = int(
        sum(
            parameter.numel()

            for parameter in (
                model.parameters()
            )
        )
    )

    (
        class_weight_tensor,
        training_class_counts,
        training_class_weights,
    ) = calculate_class_weights(
        training_df,
        device,
    )

    criterion = AblationLoss(
        ce_weight=(
            class_weight_tensor

            if condition_config[
                "use_class_weights"
            ]

            else None
        ),

        use_ordinal=(
            condition_config[
                "use_ordinal"
            ]
        ),

        ordinal_alpha=(
            condition_config[
                "ordinal_alpha"
            ]
        ),

        ordinal_loss_weight=(
            condition_config[
                "ordinal_loss_weight"
            ]
        ),
    ).to(
        device
    )

    optimizer = (
        torch.optim.AdamW(
            model.parameters(),

            lr=(
                LEARNING_RATE
            ),

            weight_decay=(
                WEIGHT_DECAY
            ),
        )
    )

    scheduler = (
        build_scheduler(
            optimizer
        )
    )

    scaler = (
        create_scaler()
    )

    start_epoch = 1

    best_epoch = -1

    best_patient_metrics = None

    best_image_metrics = None

    patience_counter = 0

    history = []

    if resume_payload is not None:
        model.load_state_dict(
            resume_payload[
                "model_state_dict"
            ]
        )

        optimizer.load_state_dict(
            resume_payload[
                "optimizer_state_dict"
            ]
        )

        optimizer_to_device(
            optimizer,
            device,
        )

        scheduler.load_state_dict(
            resume_payload[
                "scheduler_state_dict"
            ]
        )

        scaler_state = (
            resume_payload.get(
                "scaler_state_dict"
            )
        )

        if scaler_state is not None:
            scaler.load_state_dict(
                scaler_state
            )

        completed_epoch = int(
            resume_payload[
                "completed_epoch"
            ]
        )

        start_epoch = (
            completed_epoch
            +
            1
        )

        best_epoch = int(
            resume_payload[
                "best_epoch"
            ]
        )

        best_patient_metrics = (
            resume_payload[
                "best_patient_metrics"
            ]
        )

        best_image_metrics = (
            resume_payload[
                "best_image_metrics"
            ]
        )

        patience_counter = int(
            resume_payload[
                "patience_counter"
            ]
        )

        history = (
            resume_payload.get(
                "history",
                [],
            )
        )

        training_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "training_generator_state"
                ],

                "training generator",
            )
        )

        validation_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "validation_generator_state"
                ],

                "validation generator",
            )
        )

        restore_rng_state(
            resume_payload[
                "rng_state"
            ]
        )

        repair_best = False

        if not best_checkpoint_path.exists():
            repair_best = True

        else:
            try:
                existing_best = (
                    safe_torch_load(
                        best_checkpoint_path,

                        map_location="cpu",
                    )
                )

                if (
                    existing_best.get(
                        "fold_fingerprint"
                    )
                    !=
                    fold_fingerprint
                ):
                    repair_best = True

                elif (
                    int(
                        existing_best.get(
                            "best_epoch",
                            -1,
                        )
                    )
                    !=
                    best_epoch
                ):
                    repair_best = True

                del existing_best

            except Exception:
                repair_best = True

        if repair_best:
            if (
                best_epoch
                ==
                completed_epoch
            ):
                atomic_torch_save(
                    {
                        "fold_fingerprint": (
                            fold_fingerprint
                        ),

                        "condition_name": (
                            condition_name
                        ),

                        "training_seed": (
                            training_seed
                        ),

                        "fold": (
                            fold
                        ),

                        "best_epoch": (
                            best_epoch
                        ),

                        "model_state_dict": {
                            key: value
                            .detach()
                            .cpu()

                            for key, value in (
                                resume_payload[
                                    "model_state_dict"
                                ].items()
                            )
                        },

                        "best_patient_metrics": (
                            best_patient_metrics
                        ),

                        "best_image_metrics": (
                            best_image_metrics
                        ),
                    },

                    best_checkpoint_path,
                )

            else:
                raise RuntimeError(
                    "Best checkpoint missing and "
                    "cannot be reconstructed."
                )

        print(
            "Resuming from completed epoch:",
            completed_epoch,
        )

        print(
            "Next epoch:",
            start_epoch,
        )

        print(
            "Patience:",
            patience_counter,
            "/",
            EARLY_STOP_PATIENCE,
        )

    training_required = (
        start_epoch
        <=
        EPOCHS
        and
        patience_counter
        <
        EARLY_STOP_PATIENCE
    )

    if training_required:
        for epoch in range(
            start_epoch,
            EPOCHS + 1,
        ):
            training_metrics = (
                train_one_epoch(
                    model,
                    training_loader,
                    criterion,
                    optimizer,
                    scaler,
                    device,
                )
            )

            (
                validation_image_metrics,
                validation_patient_metrics,
                _,
                _,
            ) = evaluate(
                model,
                validation_loader,
                criterion,
                device,
            )

            current_lr = float(
                optimizer.param_groups[
                    0
                ][
                    "lr"
                ]
            )

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"patient QWK="
                f"{validation_patient_metrics['qwk']:.4f} | "
                f"MAE="
                f"{validation_patient_metrics['mae']:.4f} | "
                f"F1="
                f"{validation_patient_metrics['macro_f1']:.4f} | "
                f"image QWK="
                f"{validation_image_metrics['qwk']:.4f}"
            )

            improved = checkpoint_is_better(
                validation_patient_metrics,

                best_patient_metrics,

                validation_image_metrics,

                best_image_metrics,
            )

            if improved:
                best_epoch = int(
                    epoch
                )

                best_patient_metrics = (
                    copy.deepcopy(
                        validation_patient_metrics
                    )
                )

                best_image_metrics = (
                    copy.deepcopy(
                        validation_image_metrics
                    )
                )

                patience_counter = 0

                atomic_torch_save(
                    {
                        "protocol_version": (
                            PROTOCOL_VERSION
                        ),

                        "analysis_plan_fingerprint": (
                            analysis_plan_fingerprint
                        ),

                        "fold_fingerprint": (
                            fold_fingerprint
                        ),

                        "condition_name": (
                            condition_name
                        ),

                        "condition_config": (
                            condition_config
                        ),

                        "training_seed": (
                            training_seed
                        ),

                        "fold": (
                            fold
                        ),

                        "best_epoch": (
                            epoch
                        ),

                        "model_state_dict": {
                            key: value
                            .detach()
                            .cpu()

                            for key, value in (
                                model.state_dict()
                                .items()
                            )
                        },

                        "best_patient_metrics": (
                            best_patient_metrics
                        ),

                        "best_image_metrics": (
                            best_image_metrics
                        ),
                    },

                    best_checkpoint_path,
                )

                print(
                    "  New best checkpoint."
                )

            else:
                patience_counter += 1

                print(
                    "  No improvement. Patience "
                    f"{patience_counter}/"
                    f"{EARLY_STOP_PATIENCE}"
                )

            history.append({
                "epoch": (
                    epoch
                ),

                "lr": (
                    current_lr
                ),

                "training_loss": (
                    training_metrics[
                        "loss"
                    ]
                ),

                "training_image_qwk": (
                    training_metrics[
                        "qwk"
                    ]
                ),

                "validation_image_qwk": (
                    validation_image_metrics[
                        "qwk"
                    ]
                ),

                "validation_patient_qwk": (
                    validation_patient_metrics[
                        "qwk"
                    ]
                ),

                "validation_patient_mae": (
                    validation_patient_metrics[
                        "mae"
                    ]
                ),

                "validation_patient_macro_f1": (
                    validation_patient_metrics[
                        "macro_f1"
                    ]
                ),

                "validation_patient_accuracy": (
                    validation_patient_metrics[
                        "accuracy"
                    ]
                ),
            })

            atomic_csv_save(
                pd.DataFrame(
                    history
                ),

                history_path,

                index=False,
            )

            scheduler.step()

            atomic_torch_save(
                {
                    "protocol_version": (
                        PROTOCOL_VERSION
                    ),

                    "analysis_plan_fingerprint": (
                        analysis_plan_fingerprint
                    ),

                    "fold_fingerprint": (
                        fold_fingerprint
                    ),

                    "condition_name": (
                        condition_name
                    ),

                    "condition_config": (
                        condition_config
                    ),

                    "training_seed": (
                        training_seed
                    ),

                    "fold": (
                        fold
                    ),

                    "completed_epoch": (
                        epoch
                    ),

                    "best_epoch": (
                        best_epoch
                    ),

                    "best_patient_metrics": (
                        best_patient_metrics
                    ),

                    "best_image_metrics": (
                        best_image_metrics
                    ),

                    "patience_counter": (
                        patience_counter
                    ),

                    "history": (
                        history
                    ),

                    "model_state_dict": (
                        model.state_dict()
                    ),

                    "optimizer_state_dict": (
                        optimizer.state_dict()
                    ),

                    "scheduler_state_dict": (
                        scheduler.state_dict()
                    ),

                    "scaler_state_dict": (
                        scaler.state_dict()
                    ),

                    "rng_state": (
                        capture_rng_state()
                    ),

                    "training_generator_state": (
                        training_generator
                        .get_state()
                        .cpu()
                    ),

                    "validation_generator_state": (
                        validation_generator
                        .get_state()
                        .cpu()
                    ),
                },

                last_checkpoint_path,
            )

            if (
                patience_counter
                >=
                EARLY_STOP_PATIENCE
            ):
                print(
                    "Early stopping triggered."
                )

                break

    if not best_checkpoint_path.exists():
        raise RuntimeError(
            "Best checkpoint does not exist."
        )

    best_checkpoint = (
        safe_torch_load(
            best_checkpoint_path,

            map_location="cpu",
        )
    )

    if (
        best_checkpoint.get(
            "fold_fingerprint"
        )
        !=
        fold_fingerprint
    ):
        raise RuntimeError(
            "Best checkpoint fingerprint mismatch."
        )

    model.load_state_dict(
        best_checkpoint[
            "model_state_dict"
        ]
    )

    (
        final_image_metrics,
        final_patient_metrics,
        final_image_df,
        final_patient_df,
    ) = evaluate(
        model,
        validation_loader,
        criterion,
        device,
    )

    final_image_df[
        "fold"
    ] = int(
        fold
    )

    final_image_df[
        "training_seed"
    ] = int(
        training_seed
    )

    final_image_df = (
        validate_image_predictions(
            final_image_df,

            validation_df,

            expected_seed=(
                training_seed
            ),

            expected_fold=(
                fold
            ),
        )
    )

    final_patient_df[
        "fold"
    ] = int(
        fold
    )

    final_patient_df[
        "training_seed"
    ] = int(
        training_seed
    )

    atomic_csv_save(
        final_image_df,

        image_prediction_path,

        index=False,
    )

    atomic_csv_save(
        final_patient_df,

        patient_prediction_path,

        index=False,
    )

    atomic_csv_save(
        make_confusion_df(
            final_patient_df[
                "y_true"
            ],

            final_patient_df[
                "y_pred"
            ],
        ),

        fold_directory
        / "validation_patient_confusion_matrix.csv",

        index=True,
    )

    fold_result = {
        "completed_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "protocol_version": (
            PROTOCOL_VERSION
        ),

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),

        "fold_fingerprint": (
            fold_fingerprint
        ),

        "condition_name": (
            condition_name
        ),

        "condition_display_name": (
            condition_config[
                "display_name"
            ]
        ),

        "training_seed": (
            int(
                training_seed
            )
        ),

        "fold": (
            int(
                fold
            )
        ),

        "best_epoch": (
            int(
                best_checkpoint[
                    "best_epoch"
                ]
            )
        ),

        "parameter_count": (
            int(
                parameter_count
            )
        ),

        "use_class_weights": (
            bool(
                condition_config[
                    "use_class_weights"
                ]
            )
        ),

        "use_ordinal": (
            bool(
                condition_config[
                    "use_ordinal"
                ]
            )
        ),

        "ordinal_alpha": (
            float(
                condition_config[
                    "ordinal_alpha"
                ]
            )
        ),

        "ordinal_loss_weight": (
            float(
                condition_config[
                    "ordinal_loss_weight"
                ]
            )
        ),

        "training_image_class_counts": (
            training_class_counts
        ),

        "training_image_class_weights": (
            training_class_weights
        ),

        "image_metrics": {
            key: float(
                value
            )

            for key, value in (
                final_image_metrics.items()
            )
        },

        "patient_metrics": {
            key: float(
                value
            )

            for key, value in (
                final_patient_metrics.items()
            )
        },

        "official_test_loaded": False,

        "official_test_evaluated": False,
    }

    atomic_json_save(
        fold_result,

        fold_result_path,
    )

    if last_checkpoint_path.exists():
        last_checkpoint_path.unlink()

    print(
        "Completed fold:",
        condition_name,
        "| seed",
        training_seed,
        "| fold",
        fold,
    )

    del model
    del criterion
    del optimizer
    del scheduler
    del scaler
    del best_checkpoint

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "fold_result": (
            fold_result
        ),

        "image_predictions": (
            final_image_df
        ),

        "patient_predictions": (
            final_patient_df
        ),
    }


# ============================================================
# 27. Patient bootstrap
# ============================================================

def stratified_patient_bootstrap(
    patient_df,
    repetitions,
    seed,
):
    patient_df = (
        patient_df
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )

    y_true = (
        patient_df[
            "y_true"
        ]
        .to_numpy(
            dtype=int
        )
    )

    y_pred = (
        patient_df[
            "y_pred"
        ]
        .to_numpy(
            dtype=int
        )
    )

    class_indices = {
        class_index: np.flatnonzero(
            y_true
            ==
            class_index
        )

        for class_index in range(
            NUM_CLASSES
        )
    }

    for class_index, indices in (
        class_indices.items()
    ):
        if len(indices) == 0:
            raise RuntimeError(
                "Bootstrap class is empty: "
                f"{class_index}"
            )

    rng = np.random.default_rng(
        seed
    )

    rows = []

    for bootstrap_index in range(
        repetitions
    ):
        sampled_indices = np.concatenate([
            rng.choice(
                class_indices[
                    class_index
                ],

                size=len(
                    class_indices[
                        class_index
                    ]
                ),

                replace=True,
            )

            for class_index in range(
                NUM_CLASSES
            )
        ])

        metrics = compute_metrics(
            y_true[
                sampled_indices
            ],

            y_pred[
                sampled_indices
            ],
        )

        rows.append({
            "bootstrap_index": (
                bootstrap_index
            ),

            **metrics,
        })

    bootstrap_df = pd.DataFrame(
        rows
    )

    interval_rows = []

    for metric_name in [
        "qwk",
        "mae",
        "macro_f1",
        "accuracy",
    ]:
        values = (
            bootstrap_df[
                metric_name
            ]
            .dropna()
            .to_numpy()
        )

        interval_rows.append({
            "metric": (
                metric_name
            ),

            "lower_95": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

            "upper_95": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),

            "n_valid_bootstrap": (
                int(
                    len(
                        values
                    )
                )
            ),
        })

    return (
        pd.DataFrame(
            interval_rows
        ),

        bootstrap_df,
    )


# ============================================================
# 28. Load A0 full-model OOF predictions
# ============================================================

image_prediction_lookup = {
    "A0_full_locked": {},
}

patient_prediction_lookup = {
    "A0_full_locked": {},
}

per_seed_result_rows = []

for seed in SEEDS:
    image_path = (
        NOTEBOOK06_ROOT
        / "Swin"
        / f"seed_{seed}"
        / "oof_image_predictions.csv"
    )

    raw_image_df = pd.read_csv(
        image_path
    )

    image_df = (
        validate_image_predictions(
            raw_image_df,

            trainval_df,

            expected_seed=(
                seed
            ),

            expected_fold=None,
        )
    )

    if (
        "fold"
        not in
        image_df.columns
    ):
        raise RuntimeError(
            "A0 OOF predictions do not "
            "contain fold IDs."
        )

    if (
        image_df
        .groupby(
            "patient_id"
        )[
            "fold"
        ]
        .nunique()
        .max()
        !=
        1
    ):
        raise RuntimeError(
            "A0 patient appears in "
            "multiple OOF folds."
        )

    patient_df = (
        aggregate_patient_probabilities(
            image_df
        )
    )

    if len(patient_df) != 478:
        raise RuntimeError(
            "A0 patient count is not 478."
        )

    image_metrics = (
        compute_metrics(
            image_df[
                "y_true"
            ],

            image_df[
                "y_pred"
            ],
        )
    )

    patient_metrics = (
        compute_metrics(
            patient_df[
                "y_true"
            ],

            patient_df[
                "y_pred"
            ],
        )
    )

    (
        bootstrap_ci_df,
        _,
    ) = stratified_patient_bootstrap(
        patient_df,

        repetitions=(
            BOOTSTRAP_REPETITIONS
        ),

        seed=(
            BOOTSTRAP_SEED
            +
            seed
        ),
    )

    ci_lookup = (
        bootstrap_ci_df
        .set_index(
            "metric"
        )
    )

    image_prediction_lookup[
        "A0_full_locked"
    ][
        seed
    ] = (
        image_df
    )

    patient_prediction_lookup[
        "A0_full_locked"
    ][
        seed
    ] = (
        patient_df
    )

    per_seed_result_rows.append({
        "condition_name": (
            "A0_full_locked"
        ),

        "display_name": (
            ABLATION_CONFIGS[
                "A0_full_locked"
            ][
                "display_name"
            ]
        ),

        "role": (
            "locked_primary_model"
        ),

        "training_seed": (
            int(
                seed
            )
        ),

        "use_class_weights": True,

        "use_ordinal": True,

        "ordinal_loss_weight": 0.10,

        "oof_patient_qwk": (
            patient_metrics[
                "qwk"
            ]
        ),

        "oof_patient_qwk_ci_low": float(
            ci_lookup.loc[
                "qwk",
                "lower_95",
            ]
        ),

        "oof_patient_qwk_ci_high": float(
            ci_lookup.loc[
                "qwk",
                "upper_95",
            ]
        ),

        "oof_patient_mae": (
            patient_metrics[
                "mae"
            ]
        ),

        "oof_patient_macro_f1": (
            patient_metrics[
                "macro_f1"
            ]
        ),

        "oof_patient_accuracy": (
            patient_metrics[
                "accuracy"
            ]
        ),

        "oof_image_qwk": (
            image_metrics[
                "qwk"
            ]
        ),

        "oof_image_mae": (
            image_metrics[
                "mae"
            ]
        ),

        "oof_image_macro_f1": (
            image_metrics[
                "macro_f1"
            ]
        ),

        "oof_image_accuracy": (
            image_metrics[
                "accuracy"
            ]
        ),

        "trained_in_notebook07": False,

        "official_test_loaded": False,

        "official_test_evaluated": False,
    })

print(
    "\nLoaded and validated all three "
    "A0 full-model OOF sources."
)


# ============================================================
# 29. Run one complete ablation-seed experiment
# ============================================================

def run_ablation_seed(
    condition_name,
    training_seed,
):
    condition_config = (
        ABLATION_CONFIGS[
            condition_name
        ]
    )

    seed_directory = (
        OUTPUT_ROOT
        /
        condition_name
        /
        f"seed_{training_seed}"
    )

    seed_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    seed_configuration = {
        "protocol_version": (
            PROTOCOL_VERSION
        ),

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),

        "condition_name": (
            condition_name
        ),

        "condition_config": (
            condition_config
        ),

        "training_seed": (
            int(
                training_seed
            )
        ),
    }

    seed_fingerprint = (
        dictionary_sha256(
            seed_configuration
        )
    )

    seed_result_path = (
        seed_directory
        / "seed_result.json"
    )

    oof_image_path = (
        seed_directory
        / "oof_image_predictions.csv"
    )

    oof_patient_path = (
        seed_directory
        / "oof_patient_predictions.csv"
    )

    bootstrap_ci_path = (
        seed_directory
        / "oof_patient_bootstrap_ci.csv"
    )

    bootstrap_samples_path = (
        seed_directory
        / "oof_patient_bootstrap_samples.csv"
    )

    fold_summary_path = (
        seed_directory
        / "fold_summary.csv"
    )

    completed_paths = [
        seed_result_path,
        oof_image_path,
        oof_patient_path,
        bootstrap_ci_path,
        bootstrap_samples_path,
        fold_summary_path,
    ]

    if (
        SKIP_COMPLETED_SEED_EXPERIMENTS
        and
        all(
            path.exists()

            for path in (
                completed_paths
            )
        )
    ):
        with open(
            seed_result_path,
            "r",
            encoding="utf-8",
        ) as file:
            seed_result = (
                json.load(
                    file
                )
            )

        if (
            seed_result.get(
                "seed_fingerprint"
            )
            !=
            seed_fingerprint
        ):
            raise RuntimeError(
                "Completed ablation-seed "
                "fingerprint mismatch."
            )

        image_df = pd.read_csv(
            oof_image_path
        )

        image_df = (
            validate_image_predictions(
                image_df,

                trainval_df,

                expected_seed=(
                    training_seed
                ),

                expected_fold=None,
            )
        )

        patient_df = (
            aggregate_patient_probabilities(
                image_df
            )
        )

        print(
            "\nLoaded completed ablation seed:",
            condition_name,
            training_seed,
        )

        return {
            "result_row": (
                seed_result
            ),

            "oof_image_df": (
                image_df
            ),

            "oof_patient_df": (
                patient_df
            ),
        }

    fold_image_parts = []

    fold_summary_rows = []

    for fold in range(
        NUM_FOLDS
    ):
        fold_output = (
            run_ablation_fold(
                condition_name,

                condition_config,

                training_seed,

                fold,

                seed_directory,
            )
        )

        fold_image_parts.append(
            fold_output[
                "image_predictions"
            ]
        )

        fold_result = (
            fold_output[
                "fold_result"
            ]
        )

        fold_summary_rows.append({
            "condition_name": (
                condition_name
            ),

            "training_seed": (
                training_seed
            ),

            "fold": (
                fold
            ),

            "best_epoch": (
                fold_result[
                    "best_epoch"
                ]
            ),

            "patient_qwk": (
                fold_result[
                    "patient_metrics"
                ][
                    "qwk"
                ]
            ),

            "patient_mae": (
                fold_result[
                    "patient_metrics"
                ][
                    "mae"
                ]
            ),

            "patient_macro_f1": (
                fold_result[
                    "patient_metrics"
                ][
                    "macro_f1"
                ]
            ),

            "patient_accuracy": (
                fold_result[
                    "patient_metrics"
                ][
                    "accuracy"
                ]
            ),

            "image_qwk": (
                fold_result[
                    "image_metrics"
                ][
                    "qwk"
                ]
            ),
        })

    oof_image_df = (
        pd.concat(
            fold_image_parts,

            ignore_index=True,
        )
        .sort_values(
            "path"
        )
        .reset_index(
            drop=True
        )
    )

    oof_image_df = (
        validate_image_predictions(
            oof_image_df,

            trainval_df,

            expected_seed=(
                training_seed
            ),

            expected_fold=None,
        )
    )

    if len(oof_image_df) != 9590:
        raise RuntimeError(
            "OOF image count is not 9,590."
        )

    if (
        oof_image_df
        .groupby(
            "patient_id"
        )[
            "fold"
        ]
        .nunique()
        .max()
        !=
        1
    ):
        raise RuntimeError(
            "A patient appears in "
            "multiple OOF folds."
        )

    oof_patient_df = (
        aggregate_patient_probabilities(
            oof_image_df
        )
    )

    if len(oof_patient_df) != 478:
        raise RuntimeError(
            "OOF patient count is not 478."
        )

    image_metrics = (
        compute_metrics(
            oof_image_df[
                "y_true"
            ],

            oof_image_df[
                "y_pred"
            ],
        )
    )

    patient_metrics = (
        compute_metrics(
            oof_patient_df[
                "y_true"
            ],

            oof_patient_df[
                "y_pred"
            ],
        )
    )

    condition_index = (
        list(
            ABLATION_CONFIGS.keys()
        )
        .index(
            condition_name
        )
    )

    seed_index = (
        SEEDS.index(
            training_seed
        )
    )

    (
        bootstrap_ci_df,
        bootstrap_samples_df,
    ) = stratified_patient_bootstrap(
        oof_patient_df,

        repetitions=(
            BOOTSTRAP_REPETITIONS
        ),

        seed=(
            BOOTSTRAP_SEED
            +
            condition_index
            *
            10000
            +
            seed_index
            *
            1000
        ),
    )

    ci_lookup = (
        bootstrap_ci_df
        .set_index(
            "metric"
        )
    )

    atomic_csv_save(
        oof_image_df,

        oof_image_path,

        index=False,
    )

    atomic_csv_save(
        oof_patient_df,

        oof_patient_path,

        index=False,
    )

    atomic_csv_save(
        bootstrap_ci_df,

        bootstrap_ci_path,

        index=False,
    )

    atomic_csv_save(
        bootstrap_samples_df,

        bootstrap_samples_path,

        index=False,
    )

    atomic_csv_save(
        pd.DataFrame(
            fold_summary_rows
        ),

        fold_summary_path,

        index=False,
    )

    atomic_csv_save(
        make_confusion_df(
            oof_patient_df[
                "y_true"
            ],

            oof_patient_df[
                "y_pred"
            ],
        ),

        seed_directory
        / "oof_patient_confusion_matrix.csv",

        index=True,
    )

    result_row = {
        "condition_name": (
            condition_name
        ),

        "display_name": (
            condition_config[
                "display_name"
            ]
        ),

        "role": (
            condition_config[
                "role"
            ]
        ),

        "training_seed": (
            int(
                training_seed
            )
        ),

        "seed_fingerprint": (
            seed_fingerprint
        ),

        "use_class_weights": (
            bool(
                condition_config[
                    "use_class_weights"
                ]
            )
        ),

        "use_ordinal": (
            bool(
                condition_config[
                    "use_ordinal"
                ]
            )
        ),

        "ordinal_loss_weight": (
            float(
                condition_config[
                    "ordinal_loss_weight"
                ]
            )
        ),

        "oof_patient_qwk": (
            patient_metrics[
                "qwk"
            ]
        ),

        "oof_patient_qwk_ci_low": float(
            ci_lookup.loc[
                "qwk",
                "lower_95",
            ]
        ),

        "oof_patient_qwk_ci_high": float(
            ci_lookup.loc[
                "qwk",
                "upper_95",
            ]
        ),

        "oof_patient_mae": (
            patient_metrics[
                "mae"
            ]
        ),

        "oof_patient_macro_f1": (
            patient_metrics[
                "macro_f1"
            ]
        ),

        "oof_patient_accuracy": (
            patient_metrics[
                "accuracy"
            ]
        ),

        "oof_image_qwk": (
            image_metrics[
                "qwk"
            ]
        ),

        "oof_image_mae": (
            image_metrics[
                "mae"
            ]
        ),

        "oof_image_macro_f1": (
            image_metrics[
                "macro_f1"
            ]
        ),

        "oof_image_accuracy": (
            image_metrics[
                "accuracy"
            ]
        ),

        "trained_in_notebook07": True,

        "official_test_loaded": False,

        "official_test_evaluated": False,
    }

    atomic_json_save(
        result_row,

        seed_result_path,
    )

    print(
        "\nCompleted ablation seed:",
        condition_name,
        "| seed",
        training_seed,
    )

    print(
        "Patient metrics:",
        patient_metrics,
    )

    return {
        "result_row": (
            result_row
        ),

        "oof_image_df": (
            oof_image_df
        ),

        "oof_patient_df": (
            oof_patient_df
        ),
    }


# ============================================================
# 30. Train A1, A2, A3 across three seeds
# ============================================================

for condition_name in (
    ABLATIONS_TO_TRAIN
):
    image_prediction_lookup[
        condition_name
    ] = {}

    patient_prediction_lookup[
        condition_name
    ] = {}

    for seed in SEEDS:
        output = (
            run_ablation_seed(
                condition_name,
                seed,
            )
        )

        image_prediction_lookup[
            condition_name
        ][
            seed
        ] = (
            output[
                "oof_image_df"
            ]
        )

        patient_prediction_lookup[
            condition_name
        ][
            seed
        ] = (
            output[
                "oof_patient_df"
            ]
        )

        per_seed_result_rows.append(
            output[
                "result_row"
            ]
        )


# ============================================================
# 31. Save per-seed results
# ============================================================

per_seed_results_df = (
    pd.DataFrame(
        per_seed_result_rows
    )
)

condition_order = {
    "A0_full_locked": 0,

    "A1_weighted_ce_only": 1,

    "A2_unweighted_ce_plus_ordinal": 2,

    "A3_unweighted_ce_only": 3,
}

per_seed_results_df[
    "condition_order"
] = (
    per_seed_results_df[
        "condition_name"
    ]
    .map(
        condition_order
    )
)

per_seed_results_df = (
    per_seed_results_df
    .sort_values(
        [
            "condition_order",
            "training_seed",
        ]
    )
    .drop(
        columns=[
            "condition_order"
        ]
    )
    .reset_index(
        drop=True
    )
)

per_seed_results_path = (
    OUTPUT_ROOT
    / "ablation_results_by_seed.csv"
)

atomic_csv_save(
    per_seed_results_df,

    per_seed_results_path,

    index=False,
)


# ============================================================
# 32. Three-seed mean ± SD summary
# ============================================================

multiseed_summary_df = (
    per_seed_results_df
    .groupby(
        [
            "condition_name",
            "display_name",
            "role",
            "use_class_weights",
            "use_ordinal",
            "ordinal_loss_weight",
        ],

        as_index=False,
    )
    .agg(
        n_seeds=(
            "training_seed",
            "nunique",
        ),

        patient_qwk_mean=(
            "oof_patient_qwk",
            "mean",
        ),

        patient_qwk_std=(
            "oof_patient_qwk",
            "std",
        ),

        patient_mae_mean=(
            "oof_patient_mae",
            "mean",
        ),

        patient_mae_std=(
            "oof_patient_mae",
            "std",
        ),

        patient_macro_f1_mean=(
            "oof_patient_macro_f1",
            "mean",
        ),

        patient_macro_f1_std=(
            "oof_patient_macro_f1",
            "std",
        ),

        patient_accuracy_mean=(
            "oof_patient_accuracy",
            "mean",
        ),

        patient_accuracy_std=(
            "oof_patient_accuracy",
            "std",
        ),

        image_qwk_mean=(
            "oof_image_qwk",
            "mean",
        ),

        image_qwk_std=(
            "oof_image_qwk",
            "std",
        ),

        image_mae_mean=(
            "oof_image_mae",
            "mean",
        ),

        image_mae_std=(
            "oof_image_mae",
            "std",
        ),

        image_macro_f1_mean=(
            "oof_image_macro_f1",
            "mean",
        ),

        image_macro_f1_std=(
            "oof_image_macro_f1",
            "std",
        ),

        image_accuracy_mean=(
            "oof_image_accuracy",
            "mean",
        ),

        image_accuracy_std=(
            "oof_image_accuracy",
            "std",
        ),
    )
)

multiseed_summary_df[
    "condition_order"
] = (
    multiseed_summary_df[
        "condition_name"
    ]
    .map(
        condition_order
    )
)

multiseed_summary_df = (
    multiseed_summary_df
    .sort_values(
        "condition_order"
    )
    .drop(
        columns=[
            "condition_order"
        ]
    )
    .reset_index(
        drop=True
    )
)

multiseed_summary_path = (
    OUTPUT_ROOT
    / "ablation_multiseed_summary.csv"
)

atomic_csv_save(
    multiseed_summary_df,

    multiseed_summary_path,

    index=False,
)


# ============================================================
# 33. Validate patient alignment for all conditions/seeds
# ============================================================

base_patient_df = (
    patient_prediction_lookup[
        "A0_full_locked"
    ][
        SEEDS[
            0
        ]
    ]
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)

PATIENT_IDS = (
    base_patient_df[
        "patient_id"
    ]
    .to_numpy(
        dtype=int
    )
)

PATIENT_TRUE = (
    base_patient_df[
        "y_true"
    ]
    .to_numpy(
        dtype=int
    )
)

prediction_arrays = {}

for condition_name in (
    ABLATION_CONFIGS
):
    for seed in SEEDS:
        patient_df = (
            patient_prediction_lookup[
                condition_name
            ][
                seed
            ]
            .sort_values(
                "patient_id"
            )
            .reset_index(
                drop=True
            )
        )

        if not np.array_equal(
            patient_df[
                "patient_id"
            ]
            .to_numpy(
                dtype=int
            ),

            PATIENT_IDS,
        ):
            raise RuntimeError(
                "Patient alignment failure for "
                f"{condition_name}, seed {seed}."
            )

        if not np.array_equal(
            patient_df[
                "y_true"
            ]
            .to_numpy(
                dtype=int
            ),

            PATIENT_TRUE,
        ):
            raise RuntimeError(
                "Reference alignment failure for "
                f"{condition_name}, seed {seed}."
            )

        prediction_arrays[
            (
                condition_name,
                seed,
            )
        ] = (
            patient_df[
                "y_pred"
            ]
            .to_numpy(
                dtype=int
            )
        )

CLASS_INDICES = {
    class_index: np.flatnonzero(
        PATIENT_TRUE
        ==
        class_index
    )

    for class_index in range(
        NUM_CLASSES
    )
}


# ============================================================
# 34. Paired bootstrap for one predefined contrast
# ============================================================

def paired_multiseed_contrast_bootstrap(
    contrast_name,
    contrast_config,
    repetitions,
    seed,
):
    condition_a = (
        contrast_config[
            "condition_a"
        ]
    )

    condition_b = (
        contrast_config[
            "condition_b"
        ]
    )

    rng = np.random.default_rng(
        seed
    )

    rows = []

    for bootstrap_index in range(
        repetitions
    ):
        sampled_indices = np.concatenate([
            rng.choice(
                CLASS_INDICES[
                    class_index
                ],

                size=len(
                    CLASS_INDICES[
                        class_index
                    ]
                ),

                replace=True,
            )

            for class_index in range(
                NUM_CLASSES
            )
        ])

        sampled_true = (
            PATIENT_TRUE[
                sampled_indices
            ]
        )

        condition_metrics = {}

        for condition_name in [
            condition_a,
            condition_b,
        ]:
            seed_metrics = {
                "qwk": [],
                "mae": [],
                "macro_f1": [],
                "accuracy": [],
            }

            for training_seed in (
                SEEDS
            ):
                sampled_pred = (
                    prediction_arrays[
                        (
                            condition_name,
                            training_seed,
                        )
                    ][
                        sampled_indices
                    ]
                )

                metrics = (
                    compute_metrics(
                        sampled_true,
                        sampled_pred,
                    )
                )

                for metric_name in (
                    seed_metrics
                ):
                    seed_metrics[
                        metric_name
                    ].append(
                        metrics[
                            metric_name
                        ]
                    )

            condition_metrics[
                condition_name
            ] = {
                metric_name: float(
                    np.mean(
                        values
                    )
                )

                for metric_name, values in (
                    seed_metrics.items()
                )
            }

        metrics_a = (
            condition_metrics[
                condition_a
            ]
        )

        metrics_b = (
            condition_metrics[
                condition_b
            ]
        )

        rows.append({
            "bootstrap_index": (
                bootstrap_index
            ),

            "delta_qwk": (
                metrics_a[
                    "qwk"
                ]
                -
                metrics_b[
                    "qwk"
                ]
            ),

            "delta_mae": (
                metrics_a[
                    "mae"
                ]
                -
                metrics_b[
                    "mae"
                ]
            ),

            "delta_macro_f1": (
                metrics_a[
                    "macro_f1"
                ]
                -
                metrics_b[
                    "macro_f1"
                ]
            ),

            "delta_accuracy": (
                metrics_a[
                    "accuracy"
                ]
                -
                metrics_b[
                    "accuracy"
                ]
            ),
        })

    samples_df = pd.DataFrame(
        rows
    )

    summary_lookup = (
        multiseed_summary_df
        .set_index(
            "condition_name"
        )
    )

    point_columns = {
        "qwk": (
            "patient_qwk_mean"
        ),

        "mae": (
            "patient_mae_mean"
        ),

        "macro_f1": (
            "patient_macro_f1_mean"
        ),

        "accuracy": (
            "patient_accuracy_mean"
        ),
    }

    sample_columns = {
        "qwk": (
            "delta_qwk"
        ),

        "mae": (
            "delta_mae"
        ),

        "macro_f1": (
            "delta_macro_f1"
        ),

        "accuracy": (
            "delta_accuracy"
        ),
    }

    summary_rows = []

    for metric_name in [
        "qwk",
        "mae",
        "macro_f1",
        "accuracy",
    ]:
        values = (
            samples_df[
                sample_columns[
                    metric_name
                ]
            ]
            .dropna()
            .to_numpy()
        )

        point_difference = (
            float(
                summary_lookup.loc[
                    condition_a,

                    point_columns[
                        metric_name
                    ],
                ]
            )
            -
            float(
                summary_lookup.loc[
                    condition_b,

                    point_columns[
                        metric_name
                    ],
                ]
            )
        )

        probability_nonpositive = float(
            np.mean(
                values <= 0
            )
        )

        probability_nonnegative = float(
            np.mean(
                values >= 0
            )
        )

        tail_proportion = min(
            1.0,

            2.0
            *
            min(
                probability_nonpositive,

                probability_nonnegative,
            ),
        )

        summary_rows.append({
            "contrast_name": (
                contrast_name
            ),

            "contrast_interpretation": (
                contrast_config[
                    "interpretation"
                ]
            ),

            "condition_a": (
                condition_a
            ),

            "condition_b": (
                condition_b
            ),

            "difference_definition": (
                f"{condition_a} - {condition_b}"
            ),

            "metric": (
                metric_name
            ),

            "point_difference": (
                point_difference
            ),

            "difference_ci_low": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

            "difference_ci_high": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),

            "bootstrap_two_sided_tail_proportion": (
                tail_proportion
            ),

            "direction_note": (
                "Negative favors condition A"

                if metric_name
                ==
                "mae"

                else

                "Positive favors condition A"
            ),
        })

    return (
        pd.DataFrame(
            summary_rows
        ),

        samples_df,
    )


# ============================================================
# 35. Run all predefined paired contrasts
# ============================================================

contrast_summary_parts = []

contrast_samples_parts = []

for contrast_index, (
    contrast_name,
    contrast_config,
) in enumerate(
    CONTRASTS.items()
):
    print(
        "\nRunning paired bootstrap contrast:",
        contrast_name,
    )

    (
        contrast_summary_df,
        contrast_samples_df,
    ) = paired_multiseed_contrast_bootstrap(
        contrast_name=(
            contrast_name
        ),

        contrast_config=(
            contrast_config
        ),

        repetitions=(
            BOOTSTRAP_REPETITIONS
        ),

        seed=(
            BOOTSTRAP_SEED
            +
            500000
            +
            contrast_index
            *
            10000
        ),
    )

    contrast_samples_df.insert(
        0,
        "contrast_name",
        contrast_name,
    )

    contrast_summary_parts.append(
        contrast_summary_df
    )

    contrast_samples_parts.append(
        contrast_samples_df
    )

contrast_summary_df = pd.concat(
    contrast_summary_parts,

    ignore_index=True,
)

contrast_samples_df = pd.concat(
    contrast_samples_parts,

    ignore_index=True,
)

contrast_summary_path = (
    OUTPUT_ROOT
    / "ablation_predeclared_contrasts_paired_bootstrap.csv"
)

contrast_samples_path = (
    OUTPUT_ROOT
    / "ablation_predeclared_contrasts_bootstrap_samples.csv"
)

atomic_csv_save(
    contrast_summary_df,

    contrast_summary_path,

    index=False,
)

atomic_csv_save(
    contrast_samples_df,

    contrast_samples_path,

    index=False,
)


# ============================================================
# 36. Create report-ready factorial table
# ============================================================

factorial_table_df = (
    multiseed_summary_df[
        [
            "condition_name",
            "display_name",
            "role",
            "use_class_weights",
            "use_ordinal",
            "ordinal_loss_weight",

            "patient_qwk_mean",
            "patient_qwk_std",

            "patient_mae_mean",
            "patient_mae_std",

            "patient_macro_f1_mean",
            "patient_macro_f1_std",

            "patient_accuracy_mean",
            "patient_accuracy_std",

            "image_qwk_mean",
            "image_qwk_std",

            "image_mae_mean",
            "image_mae_std",
        ]
    ]
    .copy()
)

factorial_table_path = (
    OUTPUT_ROOT
    / "table_locked_swin_factorial_ablation.csv"
)

atomic_csv_save(
    factorial_table_df,

    factorial_table_path,

    index=False,
)


# ============================================================
# 37. Confirm full model remains locked
# ============================================================

FULL_MODEL_STILL_LOCKED = True

PRIMARY_MODEL_CHANGED_BY_ABLATION = False

primary_lock_confirmation = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "locked_primary_model_before_ablation": (
        "A0_full_locked"
    ),

    "locked_backbone": (
        "Swin"
    ),

    "model_name": (
        SWIN_MODEL_NAME
    ),

    "configuration": (
        "swin_L3_expected_ordinal_distance_a1_w010"
    ),

    "use_class_weights": True,

    "use_ordinal": True,

    "ordinal_alpha": 1.0,

    "ordinal_loss_weight": 0.10,

    "primary_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "patient_aggregation": (
        LOCKED_AGGREGATION_RULE
    ),

    "ablation_role": (
        "explanatory_only"
    ),

    "primary_model_changed_by_ablation": (
        PRIMARY_MODEL_CHANGED_BY_ABLATION
    ),

    "full_model_still_locked": (
        FULL_MODEL_STILL_LOCKED
    ),

    "ablation_results_permitted_to_change_model": (
        False
    ),

    "official_test_loaded": False,

    "official_test_evaluated": False,
}

primary_lock_confirmation_path = (
    OUTPUT_ROOT
    / "primary_model_lock_confirmed_after_ablation.json"
)

atomic_json_save(
    primary_lock_confirmation,

    primary_lock_confirmation_path,
)


# ============================================================
# 38. Save overall decision summary
# ============================================================

decision_summary = {
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "protocol_version": (
        PROTOCOL_VERSION
    ),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "study_role": (
        "Explanatory factorial ablation"
    ),

    "locked_primary_condition": (
        "A0_full_locked"
    ),

    "trained_conditions": (
        ABLATIONS_TO_TRAIN
    ),

    "predeclared_contrasts": (
        list(
            CONTRASTS.keys()
        )
    ),

    "primary_model_changed": False,

    "primary_model_selection_performed": False,

    "primary_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "patient_aggregation": (
        LOCKED_AGGREGATION_RULE
    ),

    "training_seeds": (
        SEEDS
    ),

    "official_test_loaded": False,

    "official_test_evaluated": False,

    "output_files": {
        "per_seed_results": str(
            per_seed_results_path
        ),

        "multiseed_summary": str(
            multiseed_summary_path
        ),

        "factorial_table": str(
            factorial_table_path
        ),

        "paired_contrasts": str(
            contrast_summary_path
        ),

        "primary_lock_confirmation": str(
            primary_lock_confirmation_path
        ),
    },
}

decision_summary_path = (
    OUTPUT_ROOT
    / "ablation_decision_summary.json"
)

atomic_json_save(
    decision_summary,

    decision_summary_path,
)


# ============================================================
# 39. Display results
# ============================================================

print(
    "\n"
    +
    "=" * 130
)

print(
    "PER-SEED LOCKED-SWIN ABLATION RESULTS"
)

print(
    "=" * 130
)

display(
    per_seed_results_df[
        [
            "condition_name",
            "display_name",
            "role",
            "training_seed",

            "use_class_weights",
            "use_ordinal",

            "oof_patient_qwk",
            "oof_patient_qwk_ci_low",
            "oof_patient_qwk_ci_high",

            "oof_patient_mae",
            "oof_patient_macro_f1",
            "oof_patient_accuracy",

            "oof_image_qwk",
            "oof_image_mae",
        ]
    ]
)


print(
    "\n"
    +
    "=" * 130
)

print(
    "THREE-SEED MEAN ± SD FACTORIAL ABLATION"
)

print(
    "=" * 130
)

display(
    factorial_table_df
)


print(
    "\n"
    +
    "=" * 130
)

print(
    "PREDECLARED PAIRED PATIENT-BOOTSTRAP CONTRASTS"
)

print(
    "=" * 130
)

display(
    contrast_summary_df
)


print(
    "\n"
    +
    "=" * 130
)

print(
    "PRIMARY MODEL LOCK STATUS"
)

print(
    "=" * 130
)

print(
    "Locked primary condition:",
    "A0_full_locked",
)

print(
    "Locked model:",
    SWIN_MODEL_NAME,
)

print(
    "Class-weighted CE:",
    True,
)

print(
    "Ordinal-distance loss:",
    True,
)

print(
    "Ordinal alpha:",
    1.0,
)

print(
    "Ordinal lambda:",
    0.10,
)

print(
    "Primary model changed by ablation:",
    False,
)

print(
    "Ablation used for model selection:",
    False,
)


# ============================================================
# 40. Final status
# ============================================================

print(
    "\n"
    +
    "=" * 130
)

print(
    "07_locked_swin_ablation_development_only COMPLETED"
)

print(
    "=" * 130
)

print(
    "Output directory:",
    OUTPUT_ROOT,
)

print(
    "Frozen ablation plan:",
    analysis_plan_path,
)

print(
    "Per-seed results:",
    per_seed_results_path,
)

print(
    "Three-seed summary:",
    multiseed_summary_path,
)

print(
    "Factorial ablation table:",
    factorial_table_path,
)

print(
    "Paired contrasts:",
    contrast_summary_path,
)

print(
    "Primary lock confirmation:",
    primary_lock_confirmation_path,
)

print(
    "Decision summary:",
    decision_summary_path,
)

print(
    "\nPrimary patient reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Locked patient aggregation:",
    LOCKED_AGGREGATION_RULE,
)

print(
    "Official LIMUC test data loaded:",
    False,
)

print(
    "Official LIMUC test data evaluated:",
    False,
)

print(
    "\nIMPORTANT:"
)

print(
    "The ablation is explanatory only."
)

print(
    "A0 remains the locked primary model regardless "
    "of whether A1, A2, or A3 achieves a numerically "
    "higher development metric."
)

print(
    "Do NOT reselect the primary model after viewing "
    "these ablation results."
)